# PyTorch Broadcasting — 64 Beginner-First Exercises

> **Working copy:** This file may contain learner answers and scratch work. Use the paired `_virgin.ipynb` notebook whenever you want a clean retry.

**Mastery target:** Repeated practice should make you able to read tensor axes, decide whether an elementwise operation is valid, predict its result shape, and prepare semantic axes deliberately for real PyTorch code.

## What you actually need to derive mentally

For everyday PyTorch, the essential skills are:

1. name what every axis means;
2. know each operand's actual shape;
3. decide whether the operation is compatible;
4. predict the result shape;
5. insert size-one axes when the current shape does not clearly express what each value belongs to.

An **aligned shape** is a temporary paper tool for difficult compatibility questions. It is not a hidden tensor PyTorch asks you to create. This notebook teaches alignment in one dedicated section and then stops requiring it everywhere.

**Expanded axes** are even more specialized. They help explain explicit `expand`, storage views, and later gradient accumulation. They are taught only after alignment and are not required for ordinary beginner broadcasting exercises.

## How to work

1. Run the setup cell after starting or restarting the kernel.
2. Work in order; concepts are introduced before they are tested.
3. Read the explicit fixture values and name the axes.
4. Predict only the fields requested by the current exercise.
5. Write the smallest direct PyTorch expression.
6. Run the private test and revisit the nearby explanation if it fails.


## Beginner glossary

You do not need prior PyTorch terminology. These words are used throughout the notebook:

- **Tensor:** a PyTorch container holding one number or a rectangular collection of numbers.
- **Shape:** a Python tuple giving the number of positions along each direction of a tensor. For example, `(2, 3)` means two rows and three columns.
- **Axis** (plural: **axes**): one direction of a tensor. Axes are numbered from left to right starting at `0`. In a matrix, axis `0` runs through rows and axis `1` runs through columns.
- **Operand:** an input used by an operator. In `a + b`, both `a` and `b` are operands.
- **Operator:** the action applied to operands, such as `+`, `-`, or `*`.
- **Elementwise operation:** an operation that combines corresponding positions. `a * b` is elementwise multiplication; `a @ b` is matrix multiplication and follows different rules.
- **Scalar:** a tensor containing one number and no axes. Its shape is `()`.
- **Vector:** a tensor with one axis. A three-value vector has shape `(3,)`.
- **Matrix:** a tensor with two axes, usually rows and columns.
- **Rank:** the number of axes. A scalar has rank zero, a vector rank one, and a matrix rank two.
- **Singleton axis:** an axis whose size is `1`. It contains one position along that direction, so broadcasting may reuse its value.
- **Broadcasting:** PyTorch's rule for making compatible operands behave as if they had the same shape before an elementwise operator is applied.
- **Compatible shapes:** shapes that PyTorch can broadcast together. When shapes are compared from the right, each size pair must be equal or one size must be `1`.
- **Aligned shape:** a paper-only comparison shape obtained by adding conceptual size-one positions on the left. It does not reshape the real tensor.
- **Expanded axis:** an axis where an aligned size `1` behaves like a different result size. Expanded-axis tuples contain axis numbers, not shape sizes.
- **View:** a tensor with different shape or stride information that still refers to the same stored data.
- **Storage:** the memory containing a tensor's actual values.
- **Stride:** the step through storage used to move one position along an axis. A broadcasted view can use stride zero to reread one stored value.
- **Dtype:** the kind of value stored, such as floating-point numbers (`DTYPE`) or Boolean values (`torch.bool`).
- **Reduction:** an operation such as `sum` or `mean` that combines several values into fewer values.
- **Batch:** a group of examples processed together.
- **Feature:** one measured or learned value belonging to an example or token.
- **Token:** one position in a sequence.
- **Attention head:** one parallel attention calculation. **Query** means the position asking for information; **key** means a candidate position it may attend to.
- **Bias or offset:** a value added to another value. A feature bias supplies one added value per feature.
- **Mask:** values indicating which positions to keep, remove, allow, or forbid. Boolean masks use `True` and `False`.
- **Embedding:** a vector of feature values representing something such as a token or position.
- **Normalization:** recentering and rescaling values using statistics such as mean and variance.
- **Capstone/final practice:** a multi-step exercise combining several earlier ideas.

Every exercise also repeats the most important definitions and states what its axes mean.


## Beginner checklist

Before an elementwise operation, ask:

1. What is each actual shape?
2. What does each axis mean?
3. Are the shapes already equal?
4. If not, is a scalar or size-one axis clearly being reused?
5. If intent is unclear, should I insert a singleton axis with `None` or `unsqueeze`?
6. Only in the alignment section: how do the shapes right-align?
7. Only in the expansion section: which aligned size-one axes are reused?


## Course map

1. Shapes before broadcasting — Exercises 001–008
2. Value reuse before formal rules — Exercises 009–016
3. Singleton axes express intent — Exercises 017–024
4. Alignment as a dedicated reasoning tool — Exercises 025–032
5. Expansion only after alignment — Exercises 033–040
6. Invalid shapes and deliberate repairs — Exercises 041–048
7. Machine-learning axis patterns — Exercises 049–056
8. Explicit APIs, storage, and capstones — Exercises 057–064


In [1]:
# Supplied private-test infrastructure: run once after starting or restarting the kernel.
import importlib.util as _fixture_importlib
from pathlib import Path as _FixturePath

import torch

DTYPE = torch.float64
_REFS = {}
_MISSING = object()

_fixture_filename = "_torch_broadcasting_fixtures.py"
_fixture_relatives = (
    _FixturePath(_fixture_filename),
    _FixturePath("notebooks") / _fixture_filename,
)
_fixture_path = next(
    (
        base / relative
        for base in (_FixturePath.cwd(), *_FixturePath.cwd().parents)
        for relative in _fixture_relatives
        if (base / relative).is_file()
    ),
    None,
)
if _fixture_path is None:
    raise FileNotFoundError(f"Could not find {_fixture_filename}. Keep it beside the notebook.")
_fixture_spec = _fixture_importlib.spec_from_file_location("_torch_broadcasting_fixtures", _fixture_path)
assert _fixture_spec is not None and _fixture_spec.loader is not None
_fixture_module = _fixture_importlib.module_from_spec(_fixture_spec)
_fixture_spec.loader.exec_module(_fixture_module)


def _register_case(key, tensors):
    _REFS[key] = _fixture_module.build_reference(key, tensors)


def _check_private_value(variable_name, key, field):
    actual = globals().get(variable_name, _MISSING)
    assert actual is not _MISSING, f"Define `{variable_name}` in the answer cell first."
    expected = _REFS[key][field]
    assert type(actual) is type(expected), f"`{variable_name}` must have type {type(expected).__name__}."
    assert actual == expected, f"`{variable_name}` is not correct; revisit the nearby reasoning steps."
    print(f"PASS: {variable_name}")


def _check_private_tensor(variable_name, key, field):
    actual = globals().get(variable_name, _MISSING)
    assert actual is not _MISSING, f"Define `{variable_name}` in the answer cell first."
    assert isinstance(actual, torch.Tensor), f"`{variable_name}` must be a torch.Tensor."
    expected = _REFS[key][field]
    assert actual.shape == expected.shape, f"`{variable_name}` has the wrong shape."
    assert actual.dtype == expected.dtype, f"`{variable_name}` has the wrong dtype."
    try:
        torch.testing.assert_close(actual, expected, rtol=1e-7, atol=1e-9)
    except AssertionError:
        raise AssertionError(f"`{variable_name}` has the right shape but incorrect values.") from None
    print(f"PASS: {variable_name}")


print("Broadcasting helpers ready. Start at Exercise 001.")


/Users/gillesmajor/dev/pytorch_master_through_exercises/.venv/lib/python3.11/site-packages/torch/_subclasses/functional_tensor.py:368: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


Broadcasting helpers ready. Start at Exercise 001.


## 1. Shapes before broadcasting

Learn scalar, vector, matrix, rank, axes, and equal-shape elementwise operations. No alignment or expansion vocabulary appears here.

This section covers Exercises 001–008.


### Exercise 001 — Scalar plus scalar

**Purpose:** Recognize a scalar tensor: one number with no axes and shape `()`.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `a + b`

**Task:** Predict both actual shapes and the result shape, then add the scalars.

**Ingredients:** A scalar tensor has no axes, rank zero, and shape `()`.

**Axis meaning in this exercise:** Both operands are scalars, so they have no axes.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Required predictions and outputs:**

- `ex001_a_shape`: a literal Python shape tuple
- `ex001_b_shape`: a literal Python shape tuple
- `ex001_out_shape`: a literal Python shape tuple
- `ex001_out`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Equal vectors add elementwise


In [2]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
a = torch.tensor(2.0, dtype=DTYPE)
b = torch.tensor(-0.5, dtype=DTYPE)
_register_case("ex001", {'a': a, 'b': b})


In [3]:
# Exercise 001: complete only the fields requested above; do not use autograd.
# Define `ex001_a_shape`.
# Define `ex001_b_shape`.
# Define `ex001_out_shape`.
# Define `ex001_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.
ex001_a_shape = ()
ex001_b_shape = ()
ex001_out_shape = ()
ex001_out = torch.tensor(1.5, dtype=DTYPE)


In [4]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex001_a_shape", "ex001", "a_shape")
_check_private_value("ex001_b_shape", "ex001", "b_shape")
_check_private_value("ex001_out_shape", "ex001", "out_shape")
_check_private_tensor("ex001_out", "ex001", "out")


PASS: ex001_a_shape
PASS: ex001_b_shape
PASS: ex001_out_shape
PASS: ex001_out


### Exercise 002 — Equal vectors add elementwise

**Purpose:** Read a vector shape and add values at matching positions.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `a + b`

**Task:** Predict shapes, then add matching vector entries.

**Ingredients:** A length-three vector has shape `(3,)`; the comma makes it a Python tuple.

**Axis meaning in this exercise:** For both vectors, axis `0` lists the three vector positions.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Required predictions and outputs:**

- `ex002_a_shape`: a literal Python shape tuple
- `ex002_b_shape`: a literal Python shape tuple
- `ex002_out_shape`: a literal Python shape tuple
- `ex002_out`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Equal vectors multiply elementwise


In [5]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
a = torch.tensor([1.0, 2.0, 3.0], dtype=DTYPE)
b = torch.tensor([10.0, 20.0, 30.0], dtype=DTYPE)
_register_case("ex002", {'a': a, 'b': b})


In [6]:
# Exercise 002: complete only the fields requested above; do not use autograd.
# Define `ex002_a_shape`.
# Define `ex002_b_shape`.
# Define `ex002_out_shape`.
# Define `ex002_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.
ex002_a_shape = (3,)
ex002_b_shape = (3,)
ex002_out_shape = (3,)
ex002_out = torch.tensor([11.0, 22.0, 33.0], dtype=DTYPE)


In [7]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex002_a_shape", "ex002", "a_shape")
_check_private_value("ex002_b_shape", "ex002", "b_shape")
_check_private_value("ex002_out_shape", "ex002", "out_shape")
_check_private_tensor("ex002_out", "ex002", "out")


PASS: ex002_a_shape
PASS: ex002_b_shape
PASS: ex002_out_shape
PASS: ex002_out


### Exercise 003 — Equal vectors multiply elementwise

**Purpose:** Learn that `*` multiplies matching positions; it is different from matrix multiplication with `@`.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `a * b`

**Task:** Predict shapes, then multiply matching vector entries.

**Ingredients:** The `*` operator multiplies corresponding positions.

**Axis meaning in this exercise:** For both vectors, axis `0` lists the three vector positions.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Required predictions and outputs:**

- `ex003_a_shape`: a literal Python shape tuple
- `ex003_b_shape`: a literal Python shape tuple
- `ex003_out_shape`: a literal Python shape tuple
- `ex003_out`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Equal matrices add elementwise


In [8]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
a = torch.tensor([2.0, -1.0, 4.0], dtype=DTYPE)
b = torch.tensor([0.5, 3.0, -2.0], dtype=DTYPE)
_register_case("ex003", {'a': a, 'b': b})


In [9]:
# Exercise 003: complete only the fields requested above; do not use autograd.
# Define `ex003_a_shape`.
# Define `ex003_b_shape`.
# Define `ex003_out_shape`.
# Define `ex003_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.
ex003_a_shape = (3,)
ex003_b_shape = (3,)
ex003_out_shape = (3,)
ex003_out = torch.tensor([1.0, -3.0, -8.0], dtype=DTYPE)


In [10]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex003_a_shape", "ex003", "a_shape")
_check_private_value("ex003_b_shape", "ex003", "b_shape")
_check_private_value("ex003_out_shape", "ex003", "out_shape")
_check_private_tensor("ex003_out", "ex003", "out")


PASS: ex003_a_shape
PASS: ex003_b_shape
PASS: ex003_out_shape
PASS: ex003_out


### Exercise 004 — Equal matrices add elementwise

**Purpose:** Read matrix rows and columns directly from an explicit tensor.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `a + b`

**Task:** Predict both matrix shapes and add matching entries.

**Ingredients:** For shape `(2, 3)`, axis `0` has two rows and axis `1` has three columns.

**Axis meaning in this exercise:** For both matrices, axis `0` means rows and axis `1` means columns.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Required predictions and outputs:**

- `ex004_a_shape`: a literal Python shape tuple
- `ex004_b_shape`: a literal Python shape tuple
- `ex004_out_shape`: a literal Python shape tuple
- `ex004_out`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Equal matrices multiply elementwise


In [11]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
a = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]], dtype=DTYPE)
b = torch.tensor([[10.0, 20.0, 30.0], [40.0, 50.0, 60.0]], dtype=DTYPE)
_register_case("ex004", {'a': a, 'b': b})


In [12]:
# Exercise 004: complete only the fields requested above; do not use autograd.
# Define `ex004_a_shape`.
# Define `ex004_b_shape`.
# Define `ex004_out_shape`.
# Define `ex004_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.
ex004_a_shape = (2, 3)
ex004_b_shape = (2, 3)
ex004_out_shape = (2, 3)
ex004_out = torch.tensor(
    [
        [11.0, 22.0, 33.0],
        [44.0, 55.0, 66.0],
    ],
    dtype=DTYPE,
)


In [13]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex004_a_shape", "ex004", "a_shape")
_check_private_value("ex004_b_shape", "ex004", "b_shape")
_check_private_value("ex004_out_shape", "ex004", "out_shape")
_check_private_tensor("ex004_out", "ex004", "out")


PASS: ex004_a_shape
PASS: ex004_b_shape
PASS: ex004_out_shape
PASS: ex004_out


### Exercise 005 — Equal matrices multiply elementwise

**Purpose:** Confirm that equal matrix shapes need no value reuse.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `a * b`

**Task:** Predict shapes, then multiply matching matrix entries.

**Ingredients:** This is elementwise multiplication, not `a @ b`.

**Axis meaning in this exercise:** For both matrices, axis `0` means rows and axis `1` means columns.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Required predictions and outputs:**

- `ex005_a_shape`: a literal Python shape tuple
- `ex005_b_shape`: a literal Python shape tuple
- `ex005_out_shape`: a literal Python shape tuple
- `ex005_out`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** One-row matrix


In [14]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
a = torch.tensor([[1.0, 2.0], [3.0, 4.0]], dtype=DTYPE)
b = torch.tensor([[5.0, 6.0], [7.0, 8.0]], dtype=DTYPE)
_register_case("ex005", {'a': a, 'b': b})


In [15]:
# Exercise 005: complete only the fields requested above; do not use autograd.
# Define `ex005_a_shape`.
# Define `ex005_b_shape`.
# Define `ex005_out_shape`.
# Define `ex005_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.
ex005_a_shape = (2, 2)
ex005_b_shape = (2, 2)
ex005_out_shape = (2, 2)
ex005_out = torch.tensor(
    [
        [5.0, 12.0],
        [21.0, 32.0],
    ],
    dtype=DTYPE,
)


In [16]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex005_a_shape", "ex005", "a_shape")
_check_private_value("ex005_b_shape", "ex005", "b_shape")
_check_private_value("ex005_out_shape", "ex005", "out_shape")
_check_private_tensor("ex005_out", "ex005", "out")


PASS: ex005_a_shape
PASS: ex005_b_shape
PASS: ex005_out_shape
PASS: ex005_out


### Exercise 006 — One-row matrix

**Purpose:** See why a one-row matrix has two axes while a vector has one axis.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `a - b`

**Task:** Predict the two-axis shapes, then subtract.

**Ingredients:** Double brackets create a matrix with one row; its shape has two entries.

**Axis meaning in this exercise:** For both one-row matrices, axis `0` means rows and axis `1` means columns.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Required predictions and outputs:**

- `ex006_a_shape`: a literal Python shape tuple
- `ex006_b_shape`: a literal Python shape tuple
- `ex006_out_shape`: a literal Python shape tuple
- `ex006_out`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** One-element vector


In [17]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
a = torch.tensor([[1.0, 2.0, 3.0]], dtype=DTYPE)
b = torch.tensor([[4.0, 5.0, 6.0]], dtype=DTYPE)
_register_case("ex006", {'a': a, 'b': b})


In [18]:
# Exercise 006: complete only the fields requested above; do not use autograd.
# Define `ex006_a_shape`.
# Define `ex006_b_shape`.
# Define `ex006_out_shape`.
# Define `ex006_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.
ex006_a_shape = (1, 3)
ex006_b_shape = (1, 3)
ex006_out_shape = (1, 3)
ex006_out = torch.tensor(
    [
        [-3.0, -3.0, -3.0],
    ],
    dtype=DTYPE,
)


In [19]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex006_a_shape", "ex006", "a_shape")
_check_private_value("ex006_b_shape", "ex006", "b_shape")
_check_private_value("ex006_out_shape", "ex006", "out_shape")
_check_private_tensor("ex006_out", "ex006", "out")


PASS: ex006_a_shape
PASS: ex006_b_shape
PASS: ex006_out_shape
PASS: ex006_out


### Exercise 007 — One-element vector

**Purpose:** Distinguish shape `(1,)` from scalar shape `()`.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `a + b`

**Task:** Predict the vector shapes and result.

**Ingredients:** Each tensor has one axis containing one element; neither tensor is a scalar.

**Axis meaning in this exercise:** For both vectors, axis `0` lists vector positions; each vector happens to contain one position.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Required predictions and outputs:**

- `ex007_a_shape`: a literal Python shape tuple
- `ex007_b_shape`: a literal Python shape tuple
- `ex007_out_shape`: a literal Python shape tuple
- `ex007_out`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Equal three-axis tensors


In [20]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
a = torch.tensor([3.0], dtype=DTYPE)
b = torch.tensor([2.0], dtype=DTYPE)
_register_case("ex007", {'a': a, 'b': b})


In [21]:
# Exercise 007: complete only the fields requested above; do not use autograd.
# Define `ex007_a_shape`.
# Define `ex007_b_shape`.
# Define `ex007_out_shape`.
# Define `ex007_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.
ex007_a_shape = (1,)
ex007_b_shape = (1,)
ex007_out_shape = (1,)
ex007_out = torch.tensor([5.0], dtype=DTYPE)

In [22]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex007_a_shape", "ex007", "a_shape")
_check_private_value("ex007_b_shape", "ex007", "b_shape")
_check_private_value("ex007_out_shape", "ex007", "out_shape")
_check_private_tensor("ex007_out", "ex007", "out")


PASS: ex007_a_shape
PASS: ex007_b_shape
PASS: ex007_out_shape
PASS: ex007_out


### Exercise 008 — Equal three-axis tensors

**Purpose:** Apply the same position-by-position addition to tensors with three axes.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `a + b`

**Task:** Predict all three axes, then add matching entries.

**Ingredients:** Name the axes `(batch, rows, columns)` for this exercise. Equal shapes still pair position by position.

**Axis meaning in this exercise:** For both tensors, axes are `(batch, rows, columns)` in that order.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Required predictions and outputs:**

- `ex008_a_shape`: a literal Python shape tuple
- `ex008_b_shape`: a literal Python shape tuple
- `ex008_out_shape`: a literal Python shape tuple
- `ex008_out`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Scalar reused across a vector


In [23]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
a = torch.tensor(
    [
        [
            [1.0, 2.0],
            [3.0, 4.0]
        ],
    ], dtype=DTYPE
)
b = torch.tensor(
    [
        [
            [10.0, 20.0],
            [30.0, 40.0]
        ]
    ], dtype=DTYPE
)
_register_case("ex008", {'a': a, 'b': b})


In [24]:
# Exercise 008: complete only the fields requested above; do not use autograd.
# Define `ex008_a_shape`.
# Define `ex008_b_shape`.
# Define `ex008_out_shape`.
# Define `ex008_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.
ex008_a_shape = (1, 2, 2)
ex008_b_shape = (1, 2, 2)
ex008_out_shape = (1, 2, 2)
ex008_out = torch.tensor([[[11.0, 22.0], [33.0, 44.0]]], dtype=DTYPE)

In [25]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex008_a_shape", "ex008", "a_shape")
_check_private_value("ex008_b_shape", "ex008", "b_shape")
_check_private_value("ex008_out_shape", "ex008", "out_shape")
_check_private_tensor("ex008_out", "ex008", "out")


PASS: ex008_a_shape
PASS: ex008_b_shape
PASS: ex008_out_shape
PASS: ex008_out


## 2. Value reuse before formal rules

See scalar and singleton values reused through visible tensors before learning PyTorch's mechanical alignment algorithm.

This section covers Exercises 009–016.


### Exercise 009 — Scalar reused across a vector

**Purpose:** See broadcasting as PyTorch reusing a smaller tensor's values before learning the formal comparison steps.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `a + b`

**Task:** Predict actual and result shapes, then add the scalar to every vector entry.

**Ingredients:** Think of the scalar as available at each vector position. Do not write aligned shapes yet.

**Axis meaning in this exercise:** `a` is a scalar with no axes; axis `0` of `b` lists vector positions.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Explicit operator-ready tensors:** Before calculating the output, write the complete tensor values PyTorch behaves as if each operand has at the common elementwise shape. These are reasoning tensors; PyTorch usually does not allocate these full copies. Define each requested field with a literal `torch.tensor(...)`, not with `expand`, `broadcast_to`, `repeat`, or the original variable. Use the operand's dtype (`DTYPE` for floating tensors and `torch.bool` for Boolean tensors).

**Required predictions and outputs:**

- `ex009_a_before_op`: an explicit `torch.tensor(...)` literal showing `a` at the common operator-ready shape
- `ex009_b_before_op`: an explicit `torch.tensor(...)` literal showing `b` at the common operator-ready shape

- `ex009_a_shape`: a literal Python shape tuple
- `ex009_b_shape`: a literal Python shape tuple
- `ex009_out_shape`: a literal Python shape tuple
- `ex009_out`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Vector times scalar


In [26]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
a = torch.tensor(10.0, dtype=DTYPE)
b = torch.tensor([1.0, 2.0, 3.0], dtype=DTYPE)
_register_case("ex009", {'a': a, 'b': b})


In [27]:
# Exercise 009: complete only the fields requested above; do not use autograd.
# Define `ex009_a_shape`.
# Define `ex009_b_shape`.
# Define `ex009_out_shape`.
# Define `ex009_out` with PyTorch tensor operations.
# Define `ex009_a_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex009_b_before_op` as an explicit `torch.tensor(...)` literal.
# Write your work below, then run the supplied test cell.
ex009_a_shape = ()
ex009_b_shape = (3,)
ex009_a_before_op = torch.tensor([10.0, 10.0, 10.0], dtype=DTYPE)
ex009_b_before_op = torch.tensor([1.0, 2.0, 3.0], dtype=DTYPE)  # doesn't need to change
ex009_out_shape = (3,)
ex009_out = torch.tensor([11.0, 12.0, 13.0], dtype=DTYPE)

In [28]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex009_a_shape", "ex009", "a_shape")
_check_private_value("ex009_b_shape", "ex009", "b_shape")
_check_private_tensor("ex009_a_before_op", "ex009", "a_before_op")
_check_private_tensor("ex009_b_before_op", "ex009", "b_before_op")
_check_private_value("ex009_out_shape", "ex009", "out_shape")
_check_private_tensor("ex009_out", "ex009", "out")


PASS: ex009_a_shape
PASS: ex009_b_shape
PASS: ex009_a_before_op
PASS: ex009_b_before_op
PASS: ex009_out_shape
PASS: ex009_out


### Exercise 010 — Vector times scalar

**Purpose:** See that a scalar is reused even when the vector is written before it.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `a * b`

**Task:** Multiply each vector entry by the scalar.

**Ingredients:** The vector is before `*` and the scalar is after it; vector values are not reversed.

**Axis meaning in this exercise:** Axis `0` of `a` lists vector positions; `b` is a scalar with no axes.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Explicit operator-ready tensors:** Before calculating the output, write the complete tensor values PyTorch behaves as if each operand has at the common elementwise shape. These are reasoning tensors; PyTorch usually does not allocate these full copies. Define each requested field with a literal `torch.tensor(...)`, not with `expand`, `broadcast_to`, `repeat`, or the original variable. Use the operand's dtype (`DTYPE` for floating tensors and `torch.bool` for Boolean tensors).

**Required predictions and outputs:**

- `ex010_a_before_op`: an explicit `torch.tensor(...)` literal showing `a` at the common operator-ready shape
- `ex010_b_before_op`: an explicit `torch.tensor(...)` literal showing `b` at the common operator-ready shape

- `ex010_a_shape`: a literal Python shape tuple
- `ex010_b_shape`: a literal Python shape tuple
- `ex010_out_shape`: a literal Python shape tuple
- `ex010_out`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Scalar reused across a matrix


In [29]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
a = torch.tensor([1.0, -2.0, 3.0], dtype=DTYPE)
b = torch.tensor(4.0, dtype=DTYPE)
_register_case("ex010", {'a': a, 'b': b})


In [30]:
# Exercise 010: complete only the fields requested above; do not use autograd.
# Define `ex010_a_shape`.
# Define `ex010_b_shape`.
# Define `ex010_out_shape`.
# Define `ex010_out` with PyTorch tensor operations.
# Define `ex010_a_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex010_b_before_op` as an explicit `torch.tensor(...)` literal.
# Write your work below, then run the supplied test cell.
ex010_a_shape = (3,)
ex010_b_shape = ()
ex010_a_before_op = torch.tensor([1.0, -2.0, 3.0], dtype=DTYPE)
ex010_b_before_op = torch.tensor([4.0, 4.0, 4.0], dtype=DTYPE)
ex010_out_shape = (3,)
ex010_out = torch.tensor([4.0, -8.0, 12.0], dtype=DTYPE)

In [31]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex010_a_shape", "ex010", "a_shape")
_check_private_value("ex010_b_shape", "ex010", "b_shape")
_check_private_tensor("ex010_a_before_op", "ex010", "a_before_op")
_check_private_tensor("ex010_b_before_op", "ex010", "b_before_op")
_check_private_value("ex010_out_shape", "ex010", "out_shape")
_check_private_tensor("ex010_out", "ex010", "out")


PASS: ex010_a_shape
PASS: ex010_b_shape
PASS: ex010_a_before_op
PASS: ex010_b_before_op
PASS: ex010_out_shape
PASS: ex010_out


### Exercise 011 — Scalar reused across a matrix

**Purpose:** Extend scalar reuse across rows and columns.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `a + b`

**Task:** Add the scalar to every matrix position.

**Ingredients:** The matrix values are explicit so you can focus on reuse rather than decoding a constructor.

**Axis meaning in this exercise:** For `a`, axis `0` means rows and axis `1` means columns; scalar `b` has no axes.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Explicit operator-ready tensors:** Before calculating the output, write the complete tensor values PyTorch behaves as if each operand has at the common elementwise shape. These are reasoning tensors; PyTorch usually does not allocate these full copies. Define each requested field with a literal `torch.tensor(...)`, not with `expand`, `broadcast_to`, `repeat`, or the original variable. Use the operand's dtype (`DTYPE` for floating tensors and `torch.bool` for Boolean tensors).

**Required predictions and outputs:**

- `ex011_a_before_op`: an explicit `torch.tensor(...)` literal showing `a` at the common operator-ready shape
- `ex011_b_before_op`: an explicit `torch.tensor(...)` literal showing `b` at the common operator-ready shape

- `ex011_a_shape`: a literal Python shape tuple
- `ex011_b_shape`: a literal Python shape tuple
- `ex011_out_shape`: a literal Python shape tuple
- `ex011_out`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Matrix minus scalar


In [32]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
a = torch.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0]], dtype=DTYPE)
b = torch.tensor(0.5, dtype=DTYPE)
_register_case("ex011", {'a': a, 'b': b})


In [33]:
# Exercise 011: complete only the fields requested above; do not use autograd.
# Define `ex011_a_shape`.
# Define `ex011_b_shape`.
# Define `ex011_out_shape`.
# Define `ex011_out` with PyTorch tensor operations.
# Define `ex011_a_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex011_b_before_op` as an explicit `torch.tensor(...)` literal.
# Write your work below, then run the supplied test cell.
ex011_a_shape = (2, 3)
ex011_b_shape = ()
ex011_a_before_op = torch.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0]], dtype=DTYPE)
ex011_b_before_op = torch.tensor([[0.5, 0.5, 0.5], [0.5, 0.5, 0.5]], dtype=DTYPE)
ex011_out_shape = (2, 3)
ex011_out = torch.tensor([[0.5, 1.5, 2.5], [3.5, 4.5, 5.5]], dtype=DTYPE)

In [34]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex011_a_shape", "ex011", "a_shape")
_check_private_value("ex011_b_shape", "ex011", "b_shape")
_check_private_tensor("ex011_a_before_op", "ex011", "a_before_op")
_check_private_tensor("ex011_b_before_op", "ex011", "b_before_op")
_check_private_value("ex011_out_shape", "ex011", "out_shape")
_check_private_tensor("ex011_out", "ex011", "out")


PASS: ex011_a_shape
PASS: ex011_b_shape
PASS: ex011_a_before_op
PASS: ex011_b_before_op
PASS: ex011_out_shape
PASS: ex011_out


### Exercise 012 — Matrix minus scalar

**Purpose:** Reverse the placement of matrix and scalar while preserving the same result shape.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `a - b`

**Task:** Subtract the scalar from every matrix entry.

**Ingredients:** Subtraction values depend on operand order, but shape compatibility does not.

**Axis meaning in this exercise:** For `a`, axis `0` means rows and axis `1` means columns; scalar `b` has no axes.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Explicit operator-ready tensors:** Before calculating the output, write the complete tensor values PyTorch behaves as if each operand has at the common elementwise shape. These are reasoning tensors; PyTorch usually does not allocate these full copies. Define each requested field with a literal `torch.tensor(...)`, not with `expand`, `broadcast_to`, `repeat`, or the original variable. Use the operand's dtype (`DTYPE` for floating tensors and `torch.bool` for Boolean tensors).

**Required predictions and outputs:**

- `ex012_a_before_op`: an explicit `torch.tensor(...)` literal showing `a` at the common operator-ready shape
- `ex012_b_before_op`: an explicit `torch.tensor(...)` literal showing `b` at the common operator-ready shape

- `ex012_a_shape`: a literal Python shape tuple
- `ex012_b_shape`: a literal Python shape tuple
- `ex012_out_shape`: a literal Python shape tuple
- `ex012_out`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** One stored value reused across a vector


In [35]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
a = torch.tensor([[2.0, 4.0], [6.0, 8.0]], dtype=DTYPE)
b = torch.tensor(1.0, dtype=DTYPE)
_register_case("ex012", {'a': a, 'b': b})


In [36]:
# Exercise 012: complete only the fields requested above; do not use autograd.
# Define `ex012_a_shape`.
# Define `ex012_b_shape`.
# Define `ex012_out_shape`.
# Define `ex012_out` with PyTorch tensor operations.
# Define `ex012_a_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex012_b_before_op` as an explicit `torch.tensor(...)` literal.
# Write your work below, then run the supplied test cell.
ex012_a_shape = (2, 2)
ex012_b_shape = ()
ex012_a_before_op = a
ex012_b_before_op = torch.tensor([[1.0, 1.0], [1.0, 1.0]], dtype=DTYPE)
ex012_out_shape = (2, 2)
ex012_out = torch.tensor([[1.0, 3.0], [5.0, 7.0]], dtype=DTYPE)

In [37]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex012_a_shape", "ex012", "a_shape")
_check_private_value("ex012_b_shape", "ex012", "b_shape")
_check_private_tensor("ex012_a_before_op", "ex012", "a_before_op")
_check_private_tensor("ex012_b_before_op", "ex012", "b_before_op")
_check_private_value("ex012_out_shape", "ex012", "out_shape")
_check_private_tensor("ex012_out", "ex012", "out")


PASS: ex012_a_shape
PASS: ex012_b_shape
PASS: ex012_a_before_op
PASS: ex012_b_before_op
PASS: ex012_out_shape
PASS: ex012_out


### Exercise 013 — One stored value reused across a vector

**Purpose:** See how a one-element vector supplies its value at every position of a longer vector.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `a + b`

**Task:** Add the one-element vector throughout the longer vector.

**Ingredients:** The compact operand has an actual axis of size `1`; this differs from scalar shape `()` even though both can be reused.

**Axis meaning in this exercise:** Axis `0` of each operand lists vector positions; `a` has only one such position.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Explicit operator-ready tensors:** Before calculating the output, write the complete tensor values PyTorch behaves as if each operand has at the common elementwise shape. These are reasoning tensors; PyTorch usually does not allocate these full copies. Define each requested field with a literal `torch.tensor(...)`, not with `expand`, `broadcast_to`, `repeat`, or the original variable. Use the operand's dtype (`DTYPE` for floating tensors and `torch.bool` for Boolean tensors).

**Required predictions and outputs:**

- `ex013_a_before_op`: an explicit `torch.tensor(...)` literal showing `a` at the common operator-ready shape
- `ex013_b_before_op`: an explicit `torch.tensor(...)` literal showing `b` at the common operator-ready shape

- `ex013_a_shape`: a literal Python shape tuple
- `ex013_b_shape`: a literal Python shape tuple
- `ex013_out_shape`: a literal Python shape tuple
- `ex013_out`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** One stored value reused across a matrix


In [38]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
a = torch.tensor([2.0], dtype=DTYPE)
b = torch.tensor([10.0, 20.0, 30.0], dtype=DTYPE)
_register_case("ex013", {'a': a, 'b': b})


In [39]:
# Exercise 013: complete only the fields requested above; do not use autograd.
# Define `ex013_a_shape`.
# Define `ex013_b_shape`.
# Define `ex013_out_shape`.
# Define `ex013_out` with PyTorch tensor operations.
# Define `ex013_a_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex013_b_before_op` as an explicit `torch.tensor(...)` literal.
# Write your work below, then run the supplied test cell.
ex013_a_shape = (1,)
ex013_b_shape = (3,)
ex013_a_before_op = torch.tensor([2.0, 2.0, 2.0], dtype=DTYPE)
ex013_b_before_op = b
ex013_out_shape = (3,)
ex013_out = torch.tensor([12.0, 22.0, 32.0], dtype=DTYPE)

In [40]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex013_a_shape", "ex013", "a_shape")
_check_private_value("ex013_b_shape", "ex013", "b_shape")
_check_private_tensor("ex013_a_before_op", "ex013", "a_before_op")
_check_private_tensor("ex013_b_before_op", "ex013", "b_before_op")
_check_private_value("ex013_out_shape", "ex013", "out_shape")
_check_private_tensor("ex013_out", "ex013", "out")


PASS: ex013_a_shape
PASS: ex013_b_shape
PASS: ex013_a_before_op
PASS: ex013_b_before_op
PASS: ex013_out_shape
PASS: ex013_out


### Exercise 014 — One stored value reused across a matrix

**Purpose:** Use an explicit `(1, 1)` tensor throughout a matrix.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `a * b`

**Task:** Multiply every matrix entry by the one stored value.

**Ingredients:** The compact tensor already has two axes, each of size `1`.

**Axis meaning in this exercise:** Both operands use axis `0` for rows and axis `1` for columns; `a` has size `1` on both axes.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Explicit operator-ready tensors:** Before calculating the output, write the complete tensor values PyTorch behaves as if each operand has at the common elementwise shape. These are reasoning tensors; PyTorch usually does not allocate these full copies. Define each requested field with a literal `torch.tensor(...)`, not with `expand`, `broadcast_to`, `repeat`, or the original variable. Use the operand's dtype (`DTYPE` for floating tensors and `torch.bool` for Boolean tensors).

**Required predictions and outputs:**

- `ex014_a_before_op`: an explicit `torch.tensor(...)` literal showing `a` at the common operator-ready shape
- `ex014_b_before_op`: an explicit `torch.tensor(...)` literal showing `b` at the common operator-ready shape

- `ex014_a_shape`: a literal Python shape tuple
- `ex014_b_shape`: a literal Python shape tuple
- `ex014_out_shape`: a literal Python shape tuple
- `ex014_out`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Scalar reused across three axes


In [41]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
a = torch.tensor([[2.0]], dtype=DTYPE)
b = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]], dtype=DTYPE)
_register_case("ex014", {'a': a, 'b': b})


In [42]:
# Exercise 014: complete only the fields requested above; do not use autograd.
# Define `ex014_a_shape`.
# Define `ex014_b_shape`.
# Define `ex014_out_shape`.
# Define `ex014_out` with PyTorch tensor operations.
# Define `ex014_a_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex014_b_before_op` as an explicit `torch.tensor(...)` literal.
# Write your work below, then run the supplied test cell.
ex014_a_shape = (1,1)
ex014_b_shape = (2, 3)
ex014_a_before_op = torch.tensor([[2.0, 2.0, 2.0], [2.0, 2.0, 2.0]], dtype=DTYPE)
ex014_b_before_op = b
ex014_out_shape = (2, 3)
ex014_out = torch.tensor([[2.0, 4.0, 6.0], [8.0, 10.0, 12.0]], dtype=DTYPE)

In [43]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex014_a_shape", "ex014", "a_shape")
_check_private_value("ex014_b_shape", "ex014", "b_shape")
_check_private_tensor("ex014_a_before_op", "ex014", "a_before_op")
_check_private_tensor("ex014_b_before_op", "ex014", "b_before_op")
_check_private_value("ex014_out_shape", "ex014", "out_shape")
_check_private_tensor("ex014_out", "ex014", "out")


PASS: ex014_a_shape
PASS: ex014_b_shape
PASS: ex014_a_before_op
PASS: ex014_b_before_op
PASS: ex014_out_shape
PASS: ex014_out


### Exercise 015 — Scalar reused across three axes

**Purpose:** See that a scalar can be added to a tensor with any number of axes.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `a + b`

**Task:** Add the scalar throughout the rank-three tensor.

**Ingredients:** Axes are `(batch, rows, columns)`. The scalar contributes no semantic axis.

**Axis meaning in this exercise:** Axes of `a` are `(batch, rows, columns)`; scalar `b` has no axes.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Explicit operator-ready tensors:** Before calculating the output, write the complete tensor values PyTorch behaves as if each operand has at the common elementwise shape. These are reasoning tensors; PyTorch usually does not allocate these full copies. Define each requested field with a literal `torch.tensor(...)`, not with `expand`, `broadcast_to`, `repeat`, or the original variable. Use the operand's dtype (`DTYPE` for floating tensors and `torch.bool` for Boolean tensors).

**Required predictions and outputs:**

- `ex015_a_before_op`: an explicit `torch.tensor(...)` literal showing `a` at the common operator-ready shape
- `ex015_b_before_op`: an explicit `torch.tensor(...)` literal showing `b` at the common operator-ready shape

- `ex015_a_shape`: a literal Python shape tuple
- `ex015_b_shape`: a literal Python shape tuple
- `ex015_out_shape`: a literal Python shape tuple
- `ex015_out`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Equal matrices require no reuse


In [44]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
a = torch.tensor([[[1.0, 2.0], [3.0, 4.0]], [[5.0, 6.0], [7.0, 8.0]]], dtype=DTYPE)
b = torch.tensor(0.25, dtype=DTYPE)
_register_case("ex015", {'a': a, 'b': b})


In [45]:
# Exercise 015: complete only the fields requested above; do not use autograd.
# Define `ex015_a_shape`.
# Define `ex015_b_shape`.
# Define `ex015_out_shape`.
# Define `ex015_out` with PyTorch tensor operations.
# Define `ex015_a_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex015_b_before_op` as an explicit `torch.tensor(...)` literal.
# Write your work below, then run the supplied test cell.
ex015_a_shape = (2, 2, 2)
ex015_b_shape = ()
ex015_a_before_op = a
ex015_b_before_op = torch.tensor([[[0.25, 0.25], [0.25, 0.25]], [[0.25, 0.25], [0.25, 0.25]]], dtype=DTYPE)
ex015_out_shape = (2, 2, 2)
ex015_out = torch.tensor([[[1.25, 2.25], [3.25, 4.25]], [[5.25, 6.25], [7.25, 8.25]]], dtype=DTYPE)

In [46]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex015_a_shape", "ex015", "a_shape")
_check_private_value("ex015_b_shape", "ex015", "b_shape")
_check_private_tensor("ex015_a_before_op", "ex015", "a_before_op")
_check_private_tensor("ex015_b_before_op", "ex015", "b_before_op")
_check_private_value("ex015_out_shape", "ex015", "out_shape")
_check_private_tensor("ex015_out", "ex015", "out")


PASS: ex015_a_shape
PASS: ex015_b_shape
PASS: ex015_a_before_op
PASS: ex015_b_before_op
PASS: ex015_out_shape
PASS: ex015_out


### Exercise 016 — Equal matrices require no reuse

**Purpose:** Compare broadcasting with the simpler case where both tensors already have the same shape.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `a + b`

**Task:** Add equal matrices and state the result shape.

**Ingredients:** Every position already has one matching value; no operand needs reuse.

**Axis meaning in this exercise:** For both matrices, axis `0` means rows and axis `1` means columns.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Explicit operator-ready tensors:** Before calculating the output, write the complete tensor values PyTorch behaves as if each operand has at the common elementwise shape. These are reasoning tensors; PyTorch usually does not allocate these full copies. Define each requested field with a literal `torch.tensor(...)`, not with `expand`, `broadcast_to`, `repeat`, or the original variable. Use the operand's dtype (`DTYPE` for floating tensors and `torch.bool` for Boolean tensors).

**Required predictions and outputs:**

- `ex016_a_before_op`: an explicit `torch.tensor(...)` literal showing `a` at the common operator-ready shape
- `ex016_b_before_op`: an explicit `torch.tensor(...)` literal showing `b` at the common operator-ready shape

- `ex016_a_shape`: a literal Python shape tuple
- `ex016_b_shape`: a literal Python shape tuple
- `ex016_out_shape`: a literal Python shape tuple
- `ex016_out`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Explicit row tensor


In [47]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
a = torch.tensor([[1.0, 2.0], [3.0, 4.0]], dtype=DTYPE)
b = torch.tensor([[10.0, 20.0], [30.0, 40.0]], dtype=DTYPE)
_register_case("ex016", {'a': a, 'b': b})


In [48]:
# Exercise 016: complete only the fields requested above; do not use autograd.
# Define `ex016_a_shape`.
# Define `ex016_b_shape`.
# Define `ex016_out_shape`.
# Define `ex016_out` with PyTorch tensor operations.
# Define `ex016_a_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex016_b_before_op` as an explicit `torch.tensor(...)` literal.
# Write your work below, then run the supplied test cell.
ex016_a_shape = (2, 2)
ex016_b_shape = (2, 2)
ex016_a_before_op = a
ex016_b_before_op = b
ex016_out_shape = (2, 2)
ex016_out = torch.tensor([[11.0, 22.0], [33.0, 44.0]], dtype=DTYPE)

In [49]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex016_a_shape", "ex016", "a_shape")
_check_private_value("ex016_b_shape", "ex016", "b_shape")
_check_private_tensor("ex016_a_before_op", "ex016", "a_before_op")
_check_private_tensor("ex016_b_before_op", "ex016", "b_before_op")
_check_private_value("ex016_out_shape", "ex016", "out_shape")
_check_private_tensor("ex016_out", "ex016", "out")


PASS: ex016_a_shape
PASS: ex016_b_shape
PASS: ex016_a_before_op
PASS: ex016_b_before_op
PASS: ex016_out_shape
PASS: ex016_out


## 3. Singleton axes express intent

Use row-shaped, column-shaped, and deliberately prepared tensors so axis meaning is visible in code.

This section covers Exercises 017–024.


### Exercise 017 — Explicit row tensor

**Purpose:** Attach three values to matrix columns with shape `(1, 3)`.

**Visible inputs:** `matrix`, `row`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `matrix + row`

**Task:** Add one value per column across both rows.

**Ingredients:** The row tensor's singleton axis `0` permits reuse down matrix rows.

**Axis meaning in this exercise:** For `matrix` and `row`, axis `0` means rows and axis `1` means columns.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Explicit operator-ready tensors:** Before calculating the output, write the complete tensor values PyTorch behaves as if each operand has at the common elementwise shape. These are reasoning tensors; PyTorch usually does not allocate these full copies. Define each requested field with a literal `torch.tensor(...)`, not with `expand`, `broadcast_to`, `repeat`, or the original variable. Use the operand's dtype (`DTYPE` for floating tensors and `torch.bool` for Boolean tensors).

**Required predictions and outputs:**

- `ex017_matrix_before_op`: an explicit `torch.tensor(...)` literal showing `matrix` at the common operator-ready shape
- `ex017_row_before_op`: an explicit `torch.tensor(...)` literal showing `row` at the common operator-ready shape

- `ex017_matrix_shape`: a literal Python shape tuple
- `ex017_row_shape`: a literal Python shape tuple
- `ex017_out_shape`: a literal Python shape tuple
- `ex017_out`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Explicit column tensor


In [50]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
matrix = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]], dtype=DTYPE)
row = torch.tensor([[10.0, 20.0, 30.0]], dtype=DTYPE)
_register_case("ex017", {'matrix': matrix, 'row': row})


In [51]:
# Exercise 017: complete only the fields requested above; do not use autograd.
# Define `ex017_matrix_shape`.
# Define `ex017_row_shape`.
# Define `ex017_out_shape`.
# Define `ex017_out` with PyTorch tensor operations.
# Define `ex017_matrix_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex017_row_before_op` as an explicit `torch.tensor(...)` literal.
# Write your work below, then run the supplied test cell.
ex017_matrix_shape = (2, 3)
ex017_row_shape = (1, 3)
ex017_matrix_before_op = matrix
ex017_row_before_op = torch.tensor([[10.0, 20.0, 30.0], [10.0, 20.0, 30.0]], dtype=DTYPE)
ex017_out_shape = (2, 3)
ex017_out = torch.tensor([[11.0, 22.0, 33.0], [14.0, 25.0, 36.0]], dtype=DTYPE)

In [52]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex017_matrix_shape", "ex017", "matrix_shape")
_check_private_value("ex017_row_shape", "ex017", "row_shape")
_check_private_tensor("ex017_matrix_before_op", "ex017", "matrix_before_op")
_check_private_tensor("ex017_row_before_op", "ex017", "row_before_op")
_check_private_value("ex017_out_shape", "ex017", "out_shape")
_check_private_tensor("ex017_out", "ex017", "out")


PASS: ex017_matrix_shape
PASS: ex017_row_shape
PASS: ex017_matrix_before_op
PASS: ex017_row_before_op
PASS: ex017_out_shape
PASS: ex017_out


### Exercise 018 — Explicit column tensor

**Purpose:** Attach two values to matrix rows with shape `(2, 1)`.

**Visible inputs:** `matrix`, `column`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `matrix + column`

**Task:** Add one value across every column of each row.

**Ingredients:** The column tensor's singleton axis `1` permits reuse across matrix columns.

**Axis meaning in this exercise:** For `matrix` and `column`, axis `0` means rows and axis `1` means columns.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Explicit operator-ready tensors:** Before calculating the output, write the complete tensor values PyTorch behaves as if each operand has at the common elementwise shape. These are reasoning tensors; PyTorch usually does not allocate these full copies. Define each requested field with a literal `torch.tensor(...)`, not with `expand`, `broadcast_to`, `repeat`, or the original variable. Use the operand's dtype (`DTYPE` for floating tensors and `torch.bool` for Boolean tensors).

**Required predictions and outputs:**

- `ex018_matrix_before_op`: an explicit `torch.tensor(...)` literal showing `matrix` at the common operator-ready shape
- `ex018_column_before_op`: an explicit `torch.tensor(...)` literal showing `column` at the common operator-ready shape

- `ex018_matrix_shape`: a literal Python shape tuple
- `ex018_column_shape`: a literal Python shape tuple
- `ex018_out_shape`: a literal Python shape tuple
- `ex018_out`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Every row-column sum


In [53]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
matrix = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]], dtype=DTYPE)
column = torch.tensor([[10.0], [20.0]], dtype=DTYPE)
_register_case("ex018", {'matrix': matrix, 'column': column})


In [54]:
# Exercise 018: complete only the fields requested above; do not use autograd.
# Define `ex018_matrix_shape`.
# Define `ex018_column_shape`.
# Define `ex018_out_shape`.
# Define `ex018_out` with PyTorch tensor operations.
# Define `ex018_matrix_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex018_column_before_op` as an explicit `torch.tensor(...)` literal.
# Write your work below, then run the supplied test cell.
ex018_matrix_shape = (2, 3)
ex018_column_shape = (2, 1)
ex018_matrix_before_op = matrix
ex018_column_before_op = torch.tensor([[10.0, 10.0, 10.0], [20.0, 20.0, 20.0]],dtype=DTYPE)
ex018_out_shape = (2, 3)
ex018_out = torch.tensor([[11.0, 12.0, 13.0], [24.0, 25.0, 26.0]], dtype=DTYPE)

In [55]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex018_matrix_shape", "ex018", "matrix_shape")
_check_private_value("ex018_column_shape", "ex018", "column_shape")
_check_private_tensor("ex018_matrix_before_op", "ex018", "matrix_before_op")
_check_private_tensor("ex018_column_before_op", "ex018", "column_before_op")
_check_private_value("ex018_out_shape", "ex018", "out_shape")
_check_private_tensor("ex018_out", "ex018", "out")


PASS: ex018_matrix_shape
PASS: ex018_column_shape
PASS: ex018_matrix_before_op
PASS: ex018_column_before_op
PASS: ex018_out_shape
PASS: ex018_out


### Exercise 019 — Every row-column sum

**Purpose:** Build a two-row, three-column result by combining a column tensor with a row tensor.

**Visible inputs:** `column`, `row`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `column + row`

**Task:** Create every column-value and row-value sum.

**Ingredients:** The column reuses values across columns; the row reuses values across rows.

**Axis meaning in this exercise:** For both tensors, axis `0` represents rows and axis `1` represents columns.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Explicit operator-ready tensors:** Before calculating the output, write the complete tensor values PyTorch behaves as if each operand has at the common elementwise shape. These are reasoning tensors; PyTorch usually does not allocate these full copies. Define each requested field with a literal `torch.tensor(...)`, not with `expand`, `broadcast_to`, `repeat`, or the original variable. Use the operand's dtype (`DTYPE` for floating tensors and `torch.bool` for Boolean tensors).

**Required predictions and outputs:**

- `ex019_column_before_op`: an explicit `torch.tensor(...)` literal showing `column` at the common operator-ready shape
- `ex019_row_before_op`: an explicit `torch.tensor(...)` literal showing `row` at the common operator-ready shape

- `ex019_column_shape`: a literal Python shape tuple
- `ex019_row_shape`: a literal Python shape tuple
- `ex019_out_shape`: a literal Python shape tuple
- `ex019_out`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Prepare row-owned values


In [68]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
column = torch.tensor([[1.0], [2.0]], dtype=DTYPE)
row = torch.tensor([[10.0, 20.0, 30.0]], dtype=DTYPE)
_register_case("ex019", {'column': column, 'row': row})


In [71]:
# Exercise 019: complete only the fields requested above; do not use autograd.
# Define `ex019_column_shape`.
# Define `ex019_row_shape`.
# Define `ex019_out_shape`.
# Define `ex019_out` with PyTorch tensor operations.
# Define `ex019_column_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex019_row_before_op` as an explicit `torch.tensor(...)` literal.
# Write your work below, then run the supplied test cell.
ex019_column_shape = (2, 1)
ex019_row_shape = (1, 3)
ex019_column_before_op = torch.tensor([[1.0, 1.0, 1.0], [2.0, 2.0, 2.0]], dtype=DTYPE)
ex019_row_before_op = torch.tensor([[10.0, 20.0, 30.0], [10.0, 20.0, 30.0]], dtype=DTYPE)
ex019_out_shape = (2, 3)
ex019_out = torch.tensor([[11.0, 21.0, 31.0], [12.0, 22.0, 32.0]], dtype=DTYPE)

In [72]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex019_column_shape", "ex019", "column_shape")
_check_private_value("ex019_row_shape", "ex019", "row_shape")
_check_private_tensor("ex019_column_before_op", "ex019", "column_before_op")
_check_private_tensor("ex019_row_before_op", "ex019", "row_before_op")
_check_private_value("ex019_out_shape", "ex019", "out_shape")
_check_private_tensor("ex019_out", "ex019", "out")


PASS: ex019_column_shape
PASS: ex019_row_shape
PASS: ex019_column_before_op
PASS: ex019_row_before_op
PASS: ex019_out_shape
PASS: ex019_out


### Exercise 020 — Prepare row-owned values

**Purpose:** Add a size-one axis at the end so the vector clearly means one value per row.

**Visible inputs:** `rows`, `grid`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `rows_col * grid`

**Task:** Create `rows_col` with `rows[:, None]`, then multiply it across columns.

**Ingredients:** Indexing with `None` inserts a real size-one axis; it does not copy values.

**Axis meaning in this exercise:** Axis `0` of `rows` means matrix rows. For `grid`, axis `0` is rows and axis `1` is columns.

**New operation or idea explained:** `:` means keep every value on the existing axis. `None` means insert a new axis of size `1`. Therefore `rows[:, None]` keeps all row values and adds one column position.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Explicit operator-ready tensors:** Before calculating the output, write the complete tensor values PyTorch behaves as if each operand has at the common elementwise shape. These are reasoning tensors; PyTorch usually does not allocate these full copies. Define each requested field with a literal `torch.tensor(...)`, not with `expand`, `broadcast_to`, `repeat`, or the original variable. Use the operand's dtype (`DTYPE` for floating tensors and `torch.bool` for Boolean tensors).

**Required predictions and outputs:**

- `ex020_rows_col_before_op`: an explicit `torch.tensor(...)` literal showing `rows_col` at the common operator-ready shape
- `ex020_grid_before_op`: an explicit `torch.tensor(...)` literal showing `grid` at the common operator-ready shape

- `ex020_rows_shape`: a literal Python shape tuple
- `ex020_grid_shape`: a literal Python shape tuple
- `ex020_rows_col_shape`: a literal Python shape tuple
- `ex020_rows_col`: the requested PyTorch tensor
- `ex020_out_shape`: a literal Python shape tuple
- `ex020_out`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Prepare column-owned values


In [73]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
rows = torch.tensor([2.0, -1.0], dtype=DTYPE)
grid = torch.tensor([[1.0, 1.0, 1.0], [1.0, 1.0, 1.0]], dtype=DTYPE)
_register_case("ex020", {'rows': rows, 'grid': grid})


In [74]:
# Exercise 020: complete only the fields requested above; do not use autograd.
# Define `ex020_rows_shape`.
# Define `ex020_grid_shape`.
# Define `ex020_rows_col_shape`.
# Define `ex020_rows_col` with PyTorch tensor operations.
# Define `ex020_out_shape`.
# Define `ex020_out` with PyTorch tensor operations.
# Define `ex020_rows_col_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex020_grid_before_op` as an explicit `torch.tensor(...)` literal.
# Write your work below, then run the supplied test cell.
ex020_rows_shape = (2,)
ex020_grid_shape = (2, 3)
ex020_rows_col_shape = (2, 1)
ex020_rows_col = torch.tensor([[2.0], [-1.0]], dtype=DTYPE)  # reshaping aka rows[:, None]
ex020_rows_col_before_op = torch.tensor([[2.0, 2.0, 2.0], [-1.0, -1.0, -1.0]], dtype=DTYPE) # broadcasting
ex020_grid_before_op = grid
ex020_out_shape = (2, 3)
ex020_out = torch.tensor([[2.0, 2.0, 2.0], [-1.0, -1.0, -1.0]], dtype=DTYPE)

In [75]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex020_rows_shape", "ex020", "rows_shape")
_check_private_value("ex020_grid_shape", "ex020", "grid_shape")
_check_private_value("ex020_rows_col_shape", "ex020", "rows_col_shape")
_check_private_tensor("ex020_rows_col", "ex020", "rows_col")
_check_private_tensor("ex020_rows_col_before_op", "ex020", "rows_col_before_op")
_check_private_tensor("ex020_grid_before_op", "ex020", "grid_before_op")
_check_private_value("ex020_out_shape", "ex020", "out_shape")
_check_private_tensor("ex020_out", "ex020", "out")


PASS: ex020_rows_shape
PASS: ex020_grid_shape
PASS: ex020_rows_col_shape
PASS: ex020_rows_col
PASS: ex020_rows_col_before_op
PASS: ex020_grid_before_op
PASS: ex020_out_shape
PASS: ex020_out


### Exercise 021 — Prepare column-owned values

**Purpose:** Add a size-one axis at the beginning so the vector clearly means one value per column.

**Visible inputs:** `cols`, `grid`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `cols_row + grid`

**Task:** Create `cols_row` with `cols[None, :]`, then add it across rows.

**Ingredients:** The inserted leading axis is real shape metadata; the following addition performs broadcasting.

**Axis meaning in this exercise:** Axis `0` of `cols` means matrix columns. For `grid`, axis `0` is rows and axis `1` is columns.

**New operation or idea explained:** `None` means insert a new axis of size `1`, while `:` keeps every existing value. Therefore `cols[None, :]` adds one row position before the column values.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Explicit operator-ready tensors:** Before calculating the output, write the complete tensor values PyTorch behaves as if each operand has at the common elementwise shape. These are reasoning tensors; PyTorch usually does not allocate these full copies. Define each requested field with a literal `torch.tensor(...)`, not with `expand`, `broadcast_to`, `repeat`, or the original variable. Use the operand's dtype (`DTYPE` for floating tensors and `torch.bool` for Boolean tensors).

**Required predictions and outputs:**

- `ex021_cols_row_before_op`: an explicit `torch.tensor(...)` literal showing `cols_row` at the common operator-ready shape
- `ex021_grid_before_op`: an explicit `torch.tensor(...)` literal showing `grid` at the common operator-ready shape

- `ex021_cols_shape`: a literal Python shape tuple
- `ex021_grid_shape`: a literal Python shape tuple
- `ex021_cols_row_shape`: a literal Python shape tuple
- `ex021_cols_row`: the requested PyTorch tensor
- `ex021_out_shape`: a literal Python shape tuple
- `ex021_out`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Unsqueeze before an axis


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
cols = torch.tensor([1.0, 10.0, 100.0], dtype=DTYPE)
grid = torch.tensor([[0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], dtype=DTYPE)
_register_case("ex021", {'cols': cols, 'grid': grid})


In [ ]:
# Exercise 021: complete only the fields requested above; do not use autograd.
# Define `ex021_cols_shape`.
# Define `ex021_grid_shape`.
# Define `ex021_cols_row_shape`.
# Define `ex021_cols_row` with PyTorch tensor operations.
# Define `ex021_out_shape`.
# Define `ex021_out` with PyTorch tensor operations.
# Define `ex021_cols_row_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex021_grid_before_op` as an explicit `torch.tensor(...)` literal.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex021_cols_shape", "ex021", "cols_shape")
_check_private_value("ex021_grid_shape", "ex021", "grid_shape")
_check_private_value("ex021_cols_row_shape", "ex021", "cols_row_shape")
_check_private_tensor("ex021_cols_row", "ex021", "cols_row")
_check_private_tensor("ex021_cols_row_before_op", "ex021", "cols_row_before_op")
_check_private_tensor("ex021_grid_before_op", "ex021", "grid_before_op")
_check_private_value("ex021_out_shape", "ex021", "out_shape")
_check_private_tensor("ex021_out", "ex021", "out")


### Exercise 022 — Unsqueeze before an axis

**Purpose:** Learn that `unsqueeze(0)` adds the same leading size-one axis as `values[None, :]`.

**Visible inputs:** `values`, `grid`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `prepared + grid`

**Task:** Create `prepared = values.unsqueeze(0)`, then add it to the grid.

**Ingredients:** `unsqueeze(0)` inserts a new axis before the original vector axis.

**Axis meaning in this exercise:** Axis `0` of `values` means columns. For `grid`, axis `0` is rows and axis `1` is columns.

**New operation or idea explained:** `unsqueeze(0)` inserts a new size-one axis at position `0`, before the vector axis. It changes shape information without copying tensor values.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Explicit operator-ready tensors:** Before calculating the output, write the complete tensor values PyTorch behaves as if each operand has at the common elementwise shape. These are reasoning tensors; PyTorch usually does not allocate these full copies. Define each requested field with a literal `torch.tensor(...)`, not with `expand`, `broadcast_to`, `repeat`, or the original variable. Use the operand's dtype (`DTYPE` for floating tensors and `torch.bool` for Boolean tensors).

**Required predictions and outputs:**

- `ex022_prepared_before_op`: an explicit `torch.tensor(...)` literal showing `prepared` at the common operator-ready shape
- `ex022_grid_before_op`: an explicit `torch.tensor(...)` literal showing `grid` at the common operator-ready shape

- `ex022_values_shape`: a literal Python shape tuple
- `ex022_grid_shape`: a literal Python shape tuple
- `ex022_prepared_shape`: a literal Python shape tuple
- `ex022_prepared`: the requested PyTorch tensor
- `ex022_out_shape`: a literal Python shape tuple
- `ex022_out`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Unsqueeze after an axis


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
values = torch.tensor([1.0, 2.0, 3.0], dtype=DTYPE)
grid = torch.tensor([[10.0, 20.0, 30.0], [40.0, 50.0, 60.0]], dtype=DTYPE)
_register_case("ex022", {'values': values, 'grid': grid})


In [ ]:
# Exercise 022: complete only the fields requested above; do not use autograd.
# Define `ex022_values_shape`.
# Define `ex022_grid_shape`.
# Define `ex022_prepared_shape`.
# Define `ex022_prepared` with PyTorch tensor operations.
# Define `ex022_out_shape`.
# Define `ex022_out` with PyTorch tensor operations.
# Define `ex022_prepared_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex022_grid_before_op` as an explicit `torch.tensor(...)` literal.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex022_values_shape", "ex022", "values_shape")
_check_private_value("ex022_grid_shape", "ex022", "grid_shape")
_check_private_value("ex022_prepared_shape", "ex022", "prepared_shape")
_check_private_tensor("ex022_prepared", "ex022", "prepared")
_check_private_tensor("ex022_prepared_before_op", "ex022", "prepared_before_op")
_check_private_tensor("ex022_grid_before_op", "ex022", "grid_before_op")
_check_private_value("ex022_out_shape", "ex022", "out_shape")
_check_private_tensor("ex022_out", "ex022", "out")


### Exercise 023 — Unsqueeze after an axis

**Purpose:** Use `unsqueeze(1)` to turn row-owned values into a column.

**Visible inputs:** `values`, `grid`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `prepared * grid`

**Task:** Create `prepared = values.unsqueeze(1)`, then multiply across columns.

**Ingredients:** Axis `1` is inserted after the existing axis `0`.

**Axis meaning in this exercise:** Axis `0` of `values` means rows. For `grid`, axis `0` is rows and axis `1` is columns.

**New operation or idea explained:** `unsqueeze(1)` inserts a new size-one axis at position `1`, after the vector axis. It changes shape information without copying tensor values.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Explicit operator-ready tensors:** Before calculating the output, write the complete tensor values PyTorch behaves as if each operand has at the common elementwise shape. These are reasoning tensors; PyTorch usually does not allocate these full copies. Define each requested field with a literal `torch.tensor(...)`, not with `expand`, `broadcast_to`, `repeat`, or the original variable. Use the operand's dtype (`DTYPE` for floating tensors and `torch.bool` for Boolean tensors).

**Required predictions and outputs:**

- `ex023_prepared_before_op`: an explicit `torch.tensor(...)` literal showing `prepared` at the common operator-ready shape
- `ex023_grid_before_op`: an explicit `torch.tensor(...)` literal showing `grid` at the common operator-ready shape

- `ex023_values_shape`: a literal Python shape tuple
- `ex023_grid_shape`: a literal Python shape tuple
- `ex023_prepared_shape`: a literal Python shape tuple
- `ex023_prepared`: the requested PyTorch tensor
- `ex023_out_shape`: a literal Python shape tuple
- `ex023_out`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Prepare image-channel values


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
values = torch.tensor([2.0, 3.0], dtype=DTYPE)
grid = torch.tensor([[1.0, 1.0, 1.0], [1.0, 1.0, 1.0]], dtype=DTYPE)
_register_case("ex023", {'values': values, 'grid': grid})


In [ ]:
# Exercise 023: complete only the fields requested above; do not use autograd.
# Define `ex023_values_shape`.
# Define `ex023_grid_shape`.
# Define `ex023_prepared_shape`.
# Define `ex023_prepared` with PyTorch tensor operations.
# Define `ex023_out_shape`.
# Define `ex023_out` with PyTorch tensor operations.
# Define `ex023_prepared_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex023_grid_before_op` as an explicit `torch.tensor(...)` literal.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex023_values_shape", "ex023", "values_shape")
_check_private_value("ex023_grid_shape", "ex023", "grid_shape")
_check_private_value("ex023_prepared_shape", "ex023", "prepared_shape")
_check_private_tensor("ex023_prepared", "ex023", "prepared")
_check_private_tensor("ex023_prepared_before_op", "ex023", "prepared_before_op")
_check_private_tensor("ex023_grid_before_op", "ex023", "grid_before_op")
_check_private_value("ex023_out_shape", "ex023", "out_shape")
_check_private_tensor("ex023_out", "ex023", "out")


### Exercise 024 — Prepare image-channel values

**Purpose:** Place a channel vector on the channel axis of `(batch, channel, height, width)`.

**Visible inputs:** `channels`, `images`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `channel_view + images`

**Task:** Create `channel_view = channels[None, :, None, None]`, then add it to every image location.

**Ingredients:** The four axes are `(batch, channel, height, width)`; only the channel axis keeps size three.

**Axis meaning in this exercise:** `channels` uses axis `0` for channels. `images` uses `(batch, channel, height, width)`.

**New operation or idea explained:** Each `None` inside brackets inserts a size-one axis. `channels[None, :, None, None]` keeps the channel values on axis `1` and creates size-one batch, height, and width axes.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Explicit operator-ready tensors:** Before calculating the output, write the complete tensor values PyTorch behaves as if each operand has at the common elementwise shape. These are reasoning tensors; PyTorch usually does not allocate these full copies. Define each requested field with a literal `torch.tensor(...)`, not with `expand`, `broadcast_to`, `repeat`, or the original variable. Use the operand's dtype (`DTYPE` for floating tensors and `torch.bool` for Boolean tensors).

**Required predictions and outputs:**

- `ex024_channel_view_before_op`: an explicit `torch.tensor(...)` literal showing `channel_view` at the common operator-ready shape
- `ex024_images_before_op`: an explicit `torch.tensor(...)` literal showing `images` at the common operator-ready shape

- `ex024_channels_shape`: a literal Python shape tuple
- `ex024_images_shape`: a literal Python shape tuple
- `ex024_channel_view_shape`: a literal Python shape tuple
- `ex024_channel_view`: the requested PyTorch tensor
- `ex024_out_shape`: a literal Python shape tuple
- `ex024_out`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Align equal vector shapes


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
channels = torch.tensor([0.1, 0.2, 0.3], dtype=DTYPE)
images = torch.zeros((2, 3, 2, 2), dtype=DTYPE)
_register_case("ex024", {'channels': channels, 'images': images})


In [ ]:
# Exercise 024: complete only the fields requested above; do not use autograd.
# Define `ex024_channels_shape`.
# Define `ex024_images_shape`.
# Define `ex024_channel_view_shape`.
# Define `ex024_channel_view` with PyTorch tensor operations.
# Define `ex024_out_shape`.
# Define `ex024_out` with PyTorch tensor operations.
# Define `ex024_channel_view_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex024_images_before_op` as an explicit `torch.tensor(...)` literal.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex024_channels_shape", "ex024", "channels_shape")
_check_private_value("ex024_images_shape", "ex024", "images_shape")
_check_private_value("ex024_channel_view_shape", "ex024", "channel_view_shape")
_check_private_tensor("ex024_channel_view", "ex024", "channel_view")
_check_private_tensor("ex024_channel_view_before_op", "ex024", "channel_view_before_op")
_check_private_tensor("ex024_images_before_op", "ex024", "images_before_op")
_check_private_value("ex024_out_shape", "ex024", "out_shape")
_check_private_tensor("ex024_out", "ex024", "out")


## 4. Alignment as a dedicated reasoning tool

Alignment is a paper method for answering one question:

> **Which axis of one operand is compared with which axis of the other operand?**

It does not create a tensor, copy values, or say that anything expands.

This section covers Exercises 025–032.

### Is mental alignment necessary?

Yes, when shapes differ or an operation fails. It is the most reliable way to debug broadcasting. You do **not** need to write aligned shapes for every obvious operation forever. These exercises isolate the skill until the procedure becomes available when needed.

### The alignment procedure

Suppose a matrix has actual shape `(3, 5)` and a vector has actual shape `(5,)`.

1. Count axes. The matrix has two axes; the vector has one.
2. Place the final dimensions under each other.
3. For comparison only, fill missing positions on the **left** with `1`.

```text
actual matrix shape:  (3, 5)
actual vector shape:     (5,)

aligned matrix shape: (3, 5)
aligned vector shape: (1, 5)
axis numbers:          0  1
```

The vector's real shape is still `(5,)`. Writing `(1, 5)` does not call `reshape`; it only records where the vector axis sits during comparison.

Now compare one aligned axis at a time:

- axis `1`: sizes `5` and `5` are equal;
- axis `0`: sizes `3` and `1` are compatible because one size is `1`.

The result uses the non-one size at each compatible axis, so this example's result shape is `(3, 5)`.

A scalar has actual shape `()`. When compared with a rank-two tensor, its conceptual aligned shape has two size-one positions. Exercises 025–032 move from equal shapes to scalars, vectors, matrices, and rank-three tensors one step at a time.

**Alignment asks where axes line up. It does not yet ask where values are reused.**


### Exercise 025 — Align equal vector shapes

**Purpose:** Practice alignment where no conceptual leading axes are needed.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `a + b`

**Task:** Write actual and aligned shapes, decide compatibility, then add.

**Ingredients:** Alignment answers where axes are compared. Equal-rank shapes remain unchanged.

**Axis meaning in this exercise:** For both vectors, axis `0` lists vector positions.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.
- **Aligned shape:** a paper-only shape used to show which axes are compared; it does not change the real tensor.

**Explicit operator-ready tensors:** Before calculating the output, write the complete tensor values PyTorch behaves as if each operand has at the common elementwise shape. These are reasoning tensors; PyTorch usually does not allocate these full copies. Define each requested field with a literal `torch.tensor(...)`, not with `expand`, `broadcast_to`, `repeat`, or the original variable. Use the operand's dtype (`DTYPE` for floating tensors and `torch.bool` for Boolean tensors).

**Required predictions and outputs:**

- `ex025_a_before_op`: an explicit `torch.tensor(...)` literal showing `a` at the common operator-ready shape
- `ex025_b_before_op`: an explicit `torch.tensor(...)` literal showing `b` at the common operator-ready shape

- `ex025_a_shape`: a literal Python shape tuple
- `ex025_a_aligned_shape`: a conceptual shape tuple after left-padding with `1`s for comparison
- `ex025_b_shape`: a literal Python shape tuple
- `ex025_b_aligned_shape`: a conceptual shape tuple after left-padding with `1`s for comparison
- `ex025_compatible`: a Python `bool` stating whether all aligned axes are compatible
- `ex025_out_shape`: a literal Python shape tuple
- `ex025_out`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Align a scalar with a vector


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
a = torch.tensor([1.0, 2.0, 3.0], dtype=DTYPE)
b = torch.tensor([4.0, 5.0, 6.0], dtype=DTYPE)
_register_case("ex025", {'a': a, 'b': b})


In [ ]:
# Exercise 025: complete only the fields requested above; do not use autograd.
# Define `ex025_a_shape`.
# Define `ex025_a_aligned_shape`.
# Define `ex025_b_shape`.
# Define `ex025_b_aligned_shape`.
# Define `ex025_compatible`.
# Define `ex025_out_shape`.
# Define `ex025_out` with PyTorch tensor operations.
# Define `ex025_a_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex025_b_before_op` as an explicit `torch.tensor(...)` literal.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex025_a_shape", "ex025", "a_shape")
_check_private_value("ex025_a_aligned_shape", "ex025", "a_aligned_shape")
_check_private_value("ex025_b_shape", "ex025", "b_shape")
_check_private_value("ex025_b_aligned_shape", "ex025", "b_aligned_shape")
_check_private_value("ex025_compatible", "ex025", "compatible")
_check_private_tensor("ex025_a_before_op", "ex025", "a_before_op")
_check_private_tensor("ex025_b_before_op", "ex025", "b_before_op")
_check_private_value("ex025_out_shape", "ex025", "out_shape")
_check_private_tensor("ex025_out", "ex025", "out")


### Exercise 026 — Align a scalar with a vector

**Purpose:** Write the paper-only comparison shape for a scalar beside a vector.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `a + b`

**Task:** Write the scalar's conceptual aligned shape and the vector's aligned shape before calculating.

**Ingredients:** Add conceptual leading `1`s only until both shapes have the same number of axes. Do not reshape either tensor.

**Axis meaning in this exercise:** The scalar has no axes; the vector's axis `0` lists vector positions.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.
- **Aligned shape:** a paper-only shape used to show which axes are compared; it does not change the real tensor.

**Explicit operator-ready tensors:** Before calculating the output, write the complete tensor values PyTorch behaves as if each operand has at the common elementwise shape. These are reasoning tensors; PyTorch usually does not allocate these full copies. Define each requested field with a literal `torch.tensor(...)`, not with `expand`, `broadcast_to`, `repeat`, or the original variable. Use the operand's dtype (`DTYPE` for floating tensors and `torch.bool` for Boolean tensors).

**Required predictions and outputs:**

- `ex026_a_before_op`: an explicit `torch.tensor(...)` literal showing `a` at the common operator-ready shape
- `ex026_b_before_op`: an explicit `torch.tensor(...)` literal showing `b` at the common operator-ready shape

- `ex026_a_shape`: a literal Python shape tuple
- `ex026_a_aligned_shape`: a conceptual shape tuple after left-padding with `1`s for comparison
- `ex026_b_shape`: a literal Python shape tuple
- `ex026_b_aligned_shape`: a conceptual shape tuple after left-padding with `1`s for comparison
- `ex026_compatible`: a Python `bool` stating whether all aligned axes are compatible
- `ex026_out_shape`: a literal Python shape tuple
- `ex026_out`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Align a vector with a matrix


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
a = torch.tensor(2.0, dtype=DTYPE)
b = torch.tensor([1.0, 2.0, 3.0], dtype=DTYPE)
_register_case("ex026", {'a': a, 'b': b})


In [ ]:
# Exercise 026: complete only the fields requested above; do not use autograd.
# Define `ex026_a_shape`.
# Define `ex026_a_aligned_shape`.
# Define `ex026_b_shape`.
# Define `ex026_b_aligned_shape`.
# Define `ex026_compatible`.
# Define `ex026_out_shape`.
# Define `ex026_out` with PyTorch tensor operations.
# Define `ex026_a_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex026_b_before_op` as an explicit `torch.tensor(...)` literal.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex026_a_shape", "ex026", "a_shape")
_check_private_value("ex026_a_aligned_shape", "ex026", "a_aligned_shape")
_check_private_value("ex026_b_shape", "ex026", "b_shape")
_check_private_value("ex026_b_aligned_shape", "ex026", "b_aligned_shape")
_check_private_value("ex026_compatible", "ex026", "compatible")
_check_private_tensor("ex026_a_before_op", "ex026", "a_before_op")
_check_private_tensor("ex026_b_before_op", "ex026", "b_before_op")
_check_private_value("ex026_out_shape", "ex026", "out_shape")
_check_private_tensor("ex026_out", "ex026", "out")


### Exercise 027 — Align a vector with a matrix

**Purpose:** See why a vector lines up with the final matrix axis.

**Visible inputs:** `matrix`, `vector`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `matrix + vector`

**Task:** Write both actual shapes, then left-pad only the shorter shape for comparison.

**Ingredients:** Alignment is bookkeeping about axis positions. It does not yet ask which axis expands.

**Axis meaning in this exercise:** `matrix` uses `(rows, columns)`; the vector values belong to columns.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.
- **Aligned shape:** a paper-only shape used to show which axes are compared; it does not change the real tensor.

**Explicit operator-ready tensors:** Before calculating the output, write the complete tensor values PyTorch behaves as if each operand has at the common elementwise shape. These are reasoning tensors; PyTorch usually does not allocate these full copies. Define each requested field with a literal `torch.tensor(...)`, not with `expand`, `broadcast_to`, `repeat`, or the original variable. Use the operand's dtype (`DTYPE` for floating tensors and `torch.bool` for Boolean tensors).

**Required predictions and outputs:**

- `ex027_matrix_before_op`: an explicit `torch.tensor(...)` literal showing `matrix` at the common operator-ready shape
- `ex027_vector_before_op`: an explicit `torch.tensor(...)` literal showing `vector` at the common operator-ready shape

- `ex027_matrix_shape`: a literal Python shape tuple
- `ex027_matrix_aligned_shape`: a conceptual shape tuple after left-padding with `1`s for comparison
- `ex027_vector_shape`: a literal Python shape tuple
- `ex027_vector_aligned_shape`: a conceptual shape tuple after left-padding with `1`s for comparison
- `ex027_compatible`: a Python `bool` stating whether all aligned axes are compatible
- `ex027_out_shape`: a literal Python shape tuple
- `ex027_out`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Align a scalar with a matrix


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
matrix = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]], dtype=DTYPE)
vector = torch.tensor([10.0, 20.0, 30.0], dtype=DTYPE)
_register_case("ex027", {'matrix': matrix, 'vector': vector})


In [ ]:
# Exercise 027: complete only the fields requested above; do not use autograd.
# Define `ex027_matrix_shape`.
# Define `ex027_matrix_aligned_shape`.
# Define `ex027_vector_shape`.
# Define `ex027_vector_aligned_shape`.
# Define `ex027_compatible`.
# Define `ex027_out_shape`.
# Define `ex027_out` with PyTorch tensor operations.
# Define `ex027_matrix_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex027_vector_before_op` as an explicit `torch.tensor(...)` literal.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex027_matrix_shape", "ex027", "matrix_shape")
_check_private_value("ex027_matrix_aligned_shape", "ex027", "matrix_aligned_shape")
_check_private_value("ex027_vector_shape", "ex027", "vector_shape")
_check_private_value("ex027_vector_aligned_shape", "ex027", "vector_aligned_shape")
_check_private_value("ex027_compatible", "ex027", "compatible")
_check_private_tensor("ex027_matrix_before_op", "ex027", "matrix_before_op")
_check_private_tensor("ex027_vector_before_op", "ex027", "vector_before_op")
_check_private_value("ex027_out_shape", "ex027", "out_shape")
_check_private_tensor("ex027_out", "ex027", "out")


### Exercise 028 — Align a scalar with a matrix

**Purpose:** Extend conceptual alignment to two matrix axes.

**Visible inputs:** `matrix`, `scalar`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `matrix + scalar`

**Task:** Align both operands to the matrix's rank and determine the result shape.

**Ingredients:** A scalar has no actual axes; conceptual leading ones fill the comparison slots.

**Axis meaning in this exercise:** `matrix` uses `(rows, columns)`; the scalar has no axes.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.
- **Aligned shape:** a paper-only shape used to show which axes are compared; it does not change the real tensor.

**Explicit operator-ready tensors:** Before calculating the output, write the complete tensor values PyTorch behaves as if each operand has at the common elementwise shape. These are reasoning tensors; PyTorch usually does not allocate these full copies. Define each requested field with a literal `torch.tensor(...)`, not with `expand`, `broadcast_to`, `repeat`, or the original variable. Use the operand's dtype (`DTYPE` for floating tensors and `torch.bool` for Boolean tensors).

**Required predictions and outputs:**

- `ex028_matrix_before_op`: an explicit `torch.tensor(...)` literal showing `matrix` at the common operator-ready shape
- `ex028_scalar_before_op`: an explicit `torch.tensor(...)` literal showing `scalar` at the common operator-ready shape

- `ex028_matrix_shape`: a literal Python shape tuple
- `ex028_matrix_aligned_shape`: a conceptual shape tuple after left-padding with `1`s for comparison
- `ex028_scalar_shape`: a literal Python shape tuple
- `ex028_scalar_aligned_shape`: a conceptual shape tuple after left-padding with `1`s for comparison
- `ex028_compatible`: a Python `bool` stating whether all aligned axes are compatible
- `ex028_out_shape`: a literal Python shape tuple
- `ex028_out`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Align a feature vector with three axes


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
matrix = torch.tensor([[1.0, 2.0], [3.0, 4.0]], dtype=DTYPE)
scalar = torch.tensor(0.5, dtype=DTYPE)
_register_case("ex028", {'matrix': matrix, 'scalar': scalar})


In [ ]:
# Exercise 028: complete only the fields requested above; do not use autograd.
# Define `ex028_matrix_shape`.
# Define `ex028_matrix_aligned_shape`.
# Define `ex028_scalar_shape`.
# Define `ex028_scalar_aligned_shape`.
# Define `ex028_compatible`.
# Define `ex028_out_shape`.
# Define `ex028_out` with PyTorch tensor operations.
# Define `ex028_matrix_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex028_scalar_before_op` as an explicit `torch.tensor(...)` literal.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex028_matrix_shape", "ex028", "matrix_shape")
_check_private_value("ex028_matrix_aligned_shape", "ex028", "matrix_aligned_shape")
_check_private_value("ex028_scalar_shape", "ex028", "scalar_shape")
_check_private_value("ex028_scalar_aligned_shape", "ex028", "scalar_aligned_shape")
_check_private_value("ex028_compatible", "ex028", "compatible")
_check_private_tensor("ex028_matrix_before_op", "ex028", "matrix_before_op")
_check_private_tensor("ex028_scalar_before_op", "ex028", "scalar_before_op")
_check_private_value("ex028_out_shape", "ex028", "out_shape")
_check_private_tensor("ex028_out", "ex028", "out")


### Exercise 029 — Align a feature vector with three axes

**Purpose:** Align features beneath the final axis of `(batch, time, features)`.

**Visible inputs:** `tokens`, `features`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `tokens + features`

**Task:** Write the three-axis aligned shapes and output shape.

**Ingredients:** Axes are `(batch, time, features)`. Right alignment places the vector on features.

**Axis meaning in this exercise:** `tokens` uses `(batch, time, features)`; the vector values belong to features.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.
- **Aligned shape:** a paper-only shape used to show which axes are compared; it does not change the real tensor.

**Explicit operator-ready tensors:** Before calculating the output, write the complete tensor values PyTorch behaves as if each operand has at the common elementwise shape. These are reasoning tensors; PyTorch usually does not allocate these full copies. Define each requested field with a literal `torch.tensor(...)`, not with `expand`, `broadcast_to`, `repeat`, or the original variable. Use the operand's dtype (`DTYPE` for floating tensors and `torch.bool` for Boolean tensors).

**Required predictions and outputs:**

- `ex029_tokens_before_op`: an explicit `torch.tensor(...)` literal showing `tokens` at the common operator-ready shape
- `ex029_features_before_op`: an explicit `torch.tensor(...)` literal showing `features` at the common operator-ready shape

- `ex029_tokens_shape`: a literal Python shape tuple
- `ex029_tokens_aligned_shape`: a conceptual shape tuple after left-padding with `1`s for comparison
- `ex029_features_shape`: a literal Python shape tuple
- `ex029_features_aligned_shape`: a conceptual shape tuple after left-padding with `1`s for comparison
- `ex029_compatible`: a Python `bool` stating whether all aligned axes are compatible
- `ex029_out_shape`: a literal Python shape tuple
- `ex029_out`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Align a matrix with three axes


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
tokens = torch.arange(24, dtype=DTYPE).reshape(2, 3, 4)
features = torch.tensor([1.0, 2.0, 3.0, 4.0], dtype=DTYPE)
_register_case("ex029", {'tokens': tokens, 'features': features})


In [ ]:
# Exercise 029: complete only the fields requested above; do not use autograd.
# Define `ex029_tokens_shape`.
# Define `ex029_tokens_aligned_shape`.
# Define `ex029_features_shape`.
# Define `ex029_features_aligned_shape`.
# Define `ex029_compatible`.
# Define `ex029_out_shape`.
# Define `ex029_out` with PyTorch tensor operations.
# Define `ex029_tokens_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex029_features_before_op` as an explicit `torch.tensor(...)` literal.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex029_tokens_shape", "ex029", "tokens_shape")
_check_private_value("ex029_tokens_aligned_shape", "ex029", "tokens_aligned_shape")
_check_private_value("ex029_features_shape", "ex029", "features_shape")
_check_private_value("ex029_features_aligned_shape", "ex029", "features_aligned_shape")
_check_private_value("ex029_compatible", "ex029", "compatible")
_check_private_tensor("ex029_tokens_before_op", "ex029", "tokens_before_op")
_check_private_tensor("ex029_features_before_op", "ex029", "features_before_op")
_check_private_value("ex029_out_shape", "ex029", "out_shape")
_check_private_tensor("ex029_out", "ex029", "out")


### Exercise 030 — Align a matrix with three axes

**Purpose:** Reuse one `(time, features)` plane across batches through right alignment.

**Visible inputs:** `tokens`, `plane`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `tokens - plane`

**Task:** Align the rank-two plane with the rank-three tensor and subtract.

**Ingredients:** The plane owns `(time, features)` and has no batch axis.

**Axis meaning in this exercise:** `tokens` uses `(batch, time, features)`; `plane` uses `(time, features)`.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.
- **Aligned shape:** a paper-only shape used to show which axes are compared; it does not change the real tensor.

**Explicit operator-ready tensors:** Before calculating the output, write the complete tensor values PyTorch behaves as if each operand has at the common elementwise shape. These are reasoning tensors; PyTorch usually does not allocate these full copies. Define each requested field with a literal `torch.tensor(...)`, not with `expand`, `broadcast_to`, `repeat`, or the original variable. Use the operand's dtype (`DTYPE` for floating tensors and `torch.bool` for Boolean tensors).

**Required predictions and outputs:**

- `ex030_tokens_before_op`: an explicit `torch.tensor(...)` literal showing `tokens` at the common operator-ready shape
- `ex030_plane_before_op`: an explicit `torch.tensor(...)` literal showing `plane` at the common operator-ready shape

- `ex030_tokens_shape`: a literal Python shape tuple
- `ex030_tokens_aligned_shape`: a conceptual shape tuple after left-padding with `1`s for comparison
- `ex030_plane_shape`: a literal Python shape tuple
- `ex030_plane_aligned_shape`: a conceptual shape tuple after left-padding with `1`s for comparison
- `ex030_compatible`: a Python `bool` stating whether all aligned axes are compatible
- `ex030_out_shape`: a literal Python shape tuple
- `ex030_out`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Alignment with singleton axes


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
tokens = torch.arange(24, dtype=DTYPE).reshape(2, 3, 4)
plane = torch.tensor([[1.0, 2.0, 3.0, 4.0], [5.0, 6.0, 7.0, 8.0], [9.0, 10.0, 11.0, 12.0]], dtype=DTYPE)
_register_case("ex030", {'tokens': tokens, 'plane': plane})


In [ ]:
# Exercise 030: complete only the fields requested above; do not use autograd.
# Define `ex030_tokens_shape`.
# Define `ex030_tokens_aligned_shape`.
# Define `ex030_plane_shape`.
# Define `ex030_plane_aligned_shape`.
# Define `ex030_compatible`.
# Define `ex030_out_shape`.
# Define `ex030_out` with PyTorch tensor operations.
# Define `ex030_tokens_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex030_plane_before_op` as an explicit `torch.tensor(...)` literal.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex030_tokens_shape", "ex030", "tokens_shape")
_check_private_value("ex030_tokens_aligned_shape", "ex030", "tokens_aligned_shape")
_check_private_value("ex030_plane_shape", "ex030", "plane_shape")
_check_private_value("ex030_plane_aligned_shape", "ex030", "plane_aligned_shape")
_check_private_value("ex030_compatible", "ex030", "compatible")
_check_private_tensor("ex030_tokens_before_op", "ex030", "tokens_before_op")
_check_private_tensor("ex030_plane_before_op", "ex030", "plane_before_op")
_check_private_value("ex030_out_shape", "ex030", "out_shape")
_check_private_tensor("ex030_out", "ex030", "out")


### Exercise 031 — Alignment with singleton axes

**Purpose:** Use alignment to compare a column and row before thinking about reuse.

**Visible inputs:** `column`, `row`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `column + row`

**Task:** Write aligned shapes, compatibility, and result shape only.

**Ingredients:** Both operands already have rank two. Alignment leaves their shapes unchanged even though their singleton axes differ.

**Axis meaning in this exercise:** For both tensors, axis `0` represents rows and axis `1` represents columns.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.
- **Aligned shape:** a paper-only shape used to show which axes are compared; it does not change the real tensor.

**Explicit operator-ready tensors:** Before calculating the output, write the complete tensor values PyTorch behaves as if each operand has at the common elementwise shape. These are reasoning tensors; PyTorch usually does not allocate these full copies. Define each requested field with a literal `torch.tensor(...)`, not with `expand`, `broadcast_to`, `repeat`, or the original variable. Use the operand's dtype (`DTYPE` for floating tensors and `torch.bool` for Boolean tensors).

**Required predictions and outputs:**

- `ex031_column_before_op`: an explicit `torch.tensor(...)` literal showing `column` at the common operator-ready shape
- `ex031_row_before_op`: an explicit `torch.tensor(...)` literal showing `row` at the common operator-ready shape

- `ex031_column_shape`: a literal Python shape tuple
- `ex031_column_aligned_shape`: a conceptual shape tuple after left-padding with `1`s for comparison
- `ex031_row_shape`: a literal Python shape tuple
- `ex031_row_aligned_shape`: a conceptual shape tuple after left-padding with `1`s for comparison
- `ex031_compatible`: a Python `bool` stating whether all aligned axes are compatible
- `ex031_out_shape`: a literal Python shape tuple
- `ex031_out`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Complete alignment check


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
column = torch.tensor([[1.0], [2.0]], dtype=DTYPE)
row = torch.tensor([[10.0, 20.0, 30.0]], dtype=DTYPE)
_register_case("ex031", {'column': column, 'row': row})


In [ ]:
# Exercise 031: complete only the fields requested above; do not use autograd.
# Define `ex031_column_shape`.
# Define `ex031_column_aligned_shape`.
# Define `ex031_row_shape`.
# Define `ex031_row_aligned_shape`.
# Define `ex031_compatible`.
# Define `ex031_out_shape`.
# Define `ex031_out` with PyTorch tensor operations.
# Define `ex031_column_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex031_row_before_op` as an explicit `torch.tensor(...)` literal.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex031_column_shape", "ex031", "column_shape")
_check_private_value("ex031_column_aligned_shape", "ex031", "column_aligned_shape")
_check_private_value("ex031_row_shape", "ex031", "row_shape")
_check_private_value("ex031_row_aligned_shape", "ex031", "row_aligned_shape")
_check_private_value("ex031_compatible", "ex031", "compatible")
_check_private_tensor("ex031_column_before_op", "ex031", "column_before_op")
_check_private_tensor("ex031_row_before_op", "ex031", "row_before_op")
_check_private_value("ex031_out_shape", "ex031", "out_shape")
_check_private_tensor("ex031_out", "ex031", "out")


### Exercise 032 — Complete alignment check

**Purpose:** Practice the complete alignment procedure without asking about expanded axes yet.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `a + b`

**Task:** Right-align both operands, determine compatibility and result shape, then add.

**Ingredients:** Work from the final axis leftward. Equal sizes or a size `1` are compatible.

**Axis meaning in this exercise:** Treat the three result axes as `(batch, rows, columns)`; `b` supplies column values.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.
- **Aligned shape:** a paper-only shape used to show which axes are compared; it does not change the real tensor.

**Explicit operator-ready tensors:** Before calculating the output, write the complete tensor values PyTorch behaves as if each operand has at the common elementwise shape. These are reasoning tensors; PyTorch usually does not allocate these full copies. Define each requested field with a literal `torch.tensor(...)`, not with `expand`, `broadcast_to`, `repeat`, or the original variable. Use the operand's dtype (`DTYPE` for floating tensors and `torch.bool` for Boolean tensors).

**Required predictions and outputs:**

- `ex032_a_before_op`: an explicit `torch.tensor(...)` literal showing `a` at the common operator-ready shape
- `ex032_b_before_op`: an explicit `torch.tensor(...)` literal showing `b` at the common operator-ready shape

- `ex032_a_shape`: a literal Python shape tuple
- `ex032_a_aligned_shape`: a conceptual shape tuple after left-padding with `1`s for comparison
- `ex032_b_shape`: a literal Python shape tuple
- `ex032_b_aligned_shape`: a conceptual shape tuple after left-padding with `1`s for comparison
- `ex032_compatible`: a Python `bool` stating whether all aligned axes are compatible
- `ex032_out_shape`: a literal Python shape tuple
- `ex032_out`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Aligned but not expanded


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
a = torch.tensor([[[1.0], [2.0], [3.0]]], dtype=DTYPE)
b = torch.tensor([10.0, 20.0, 30.0, 40.0], dtype=DTYPE)
_register_case("ex032", {'a': a, 'b': b})


In [ ]:
# Exercise 032: complete only the fields requested above; do not use autograd.
# Define `ex032_a_shape`.
# Define `ex032_a_aligned_shape`.
# Define `ex032_b_shape`.
# Define `ex032_b_aligned_shape`.
# Define `ex032_compatible`.
# Define `ex032_out_shape`.
# Define `ex032_out` with PyTorch tensor operations.
# Define `ex032_a_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex032_b_before_op` as an explicit `torch.tensor(...)` literal.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex032_a_shape", "ex032", "a_shape")
_check_private_value("ex032_a_aligned_shape", "ex032", "a_aligned_shape")
_check_private_value("ex032_b_shape", "ex032", "b_shape")
_check_private_value("ex032_b_aligned_shape", "ex032", "b_aligned_shape")
_check_private_value("ex032_compatible", "ex032", "compatible")
_check_private_tensor("ex032_a_before_op", "ex032", "a_before_op")
_check_private_tensor("ex032_b_before_op", "ex032", "b_before_op")
_check_private_value("ex032_out_shape", "ex032", "out_shape")
_check_private_tensor("ex032_out", "ex032", "out")


## 5. Expansion only after alignment

Expansion is a separate second question:

> **After the result shape is known, where must one stored value be reused?**

This section covers Exercises 033–040.

### Alignment and expansion are different data

Reuse the matrix-vector example from the previous section:

```text
aligned matrix shape: (3, 5)
aligned vector shape: (1, 5)
result shape:         (3, 5)
axis numbers:          0  1
```

Compare each aligned operand with the result:

- Matrix: aligned `(3, 5)` already equals result `(3, 5)`, so it expands on no axes: `()`.
- Vector: aligned `(1, 5)` must behave like result `(3, 5)`. Its size `1` at result axis `0` is reused for three rows, so its expanded-axes tuple is `(0,)`.

The two requested values have different meanings:

```text
aligned shape:  (1, 5)   # dimension sizes
expanded axes:  (0,)     # axis numbers
```

`(3, 5)` cannot be an expanded-axes answer because those are sizes, not axis indices. `()` means no axis expands.

PyTorch usually does not allocate a fully copied tensor first. Expansion means that the elementwise computation behaves as if size-one values were repeated. Most everyday broadcasting code requires predicting compatibility and result shape; expanded-axis audits are mainly useful for debugging, explicit `expand`, storage views, and later gradient reasoning.

Exercise 033 first demonstrates aligned tensors with no expansion. Exercises 034–040 then add one reuse pattern at a time.


### Exercise 033 — Aligned but not expanded

**Purpose:** See clearly that alignment and expansion are different steps.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `a + b`

**Task:** Write aligned shapes first, then record which axes actually reuse size-one values.

**Ingredients:** Both tensors align, but neither aligned shape contains a `1` that changes in the result.

**Axis meaning in this exercise:** For both vectors, axis `0` lists vector positions.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.
- **Aligned shape:** a paper-only shape used to show which axes are compared; it does not change the real tensor.
- **Expanded axes:** axis numbers where an aligned size `1` is reused to match the result shape; this is not another shape tuple.

**Explicit operator-ready tensors:** Before calculating the output, write the complete tensor values PyTorch behaves as if each operand has at the common elementwise shape. These are reasoning tensors; PyTorch usually does not allocate these full copies. Define each requested field with a literal `torch.tensor(...)`, not with `expand`, `broadcast_to`, `repeat`, or the original variable. Use the operand's dtype (`DTYPE` for floating tensors and `torch.bool` for Boolean tensors).

**Required predictions and outputs:**

- `ex033_a_before_op`: an explicit `torch.tensor(...)` literal showing `a` at the common operator-ready shape
- `ex033_b_before_op`: an explicit `torch.tensor(...)` literal showing `b` at the common operator-ready shape

- `ex033_a_shape`: a literal Python shape tuple
- `ex033_a_aligned_shape`: a conceptual shape tuple after left-padding with `1`s for comparison
- `ex033_a_expanded_axes`: a tuple of zero-based axis numbers where size-one values are reused
- `ex033_b_shape`: a literal Python shape tuple
- `ex033_b_aligned_shape`: a conceptual shape tuple after left-padding with `1`s for comparison
- `ex033_b_expanded_axes`: a tuple of zero-based axis numbers where size-one values are reused
- `ex033_compatible`: a Python `bool` stating whether all aligned axes are compatible
- `ex033_out_shape`: a literal Python shape tuple
- `ex033_out`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Scalar expansion over a vector


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
a = torch.tensor([1.0, 2.0, 3.0], dtype=DTYPE)
b = torch.tensor([4.0, 5.0, 6.0], dtype=DTYPE)
_register_case("ex033", {'a': a, 'b': b})


In [ ]:
# Exercise 033: complete only the fields requested above; do not use autograd.
# Define `ex033_a_shape`.
# Define `ex033_a_aligned_shape`.
# Define `ex033_a_expanded_axes`.
# Define `ex033_b_shape`.
# Define `ex033_b_aligned_shape`.
# Define `ex033_b_expanded_axes`.
# Define `ex033_compatible`.
# Define `ex033_out_shape`.
# Define `ex033_out` with PyTorch tensor operations.
# Define `ex033_a_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex033_b_before_op` as an explicit `torch.tensor(...)` literal.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex033_a_shape", "ex033", "a_shape")
_check_private_value("ex033_a_aligned_shape", "ex033", "a_aligned_shape")
_check_private_value("ex033_a_expanded_axes", "ex033", "a_expanded_axes")
_check_private_value("ex033_b_shape", "ex033", "b_shape")
_check_private_value("ex033_b_aligned_shape", "ex033", "b_aligned_shape")
_check_private_value("ex033_b_expanded_axes", "ex033", "b_expanded_axes")
_check_private_value("ex033_compatible", "ex033", "compatible")
_check_private_tensor("ex033_a_before_op", "ex033", "a_before_op")
_check_private_tensor("ex033_b_before_op", "ex033", "b_before_op")
_check_private_value("ex033_out_shape", "ex033", "out_shape")
_check_private_tensor("ex033_out", "ex033", "out")


### Exercise 034 — Scalar expansion over a vector

**Purpose:** Identify reuse only after alignment and result-shape prediction.

**Visible inputs:** `scalar`, `vector`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `scalar * vector`

**Task:** Align first; then compare each aligned shape with the result and list expanded axis numbers.

**Ingredients:** Aligned shapes contain dimension sizes. Expanded-axes tuples contain axis indices.

**Axis meaning in this exercise:** The scalar has no axes; the vector's axis `0` lists vector positions.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.
- **Aligned shape:** a paper-only shape used to show which axes are compared; it does not change the real tensor.
- **Expanded axes:** axis numbers where an aligned size `1` is reused to match the result shape; this is not another shape tuple.

**Explicit operator-ready tensors:** Before calculating the output, write the complete tensor values PyTorch behaves as if each operand has at the common elementwise shape. These are reasoning tensors; PyTorch usually does not allocate these full copies. Define each requested field with a literal `torch.tensor(...)`, not with `expand`, `broadcast_to`, `repeat`, or the original variable. Use the operand's dtype (`DTYPE` for floating tensors and `torch.bool` for Boolean tensors).

**Required predictions and outputs:**

- `ex034_scalar_before_op`: an explicit `torch.tensor(...)` literal showing `scalar` at the common operator-ready shape
- `ex034_vector_before_op`: an explicit `torch.tensor(...)` literal showing `vector` at the common operator-ready shape

- `ex034_scalar_shape`: a literal Python shape tuple
- `ex034_scalar_aligned_shape`: a conceptual shape tuple after left-padding with `1`s for comparison
- `ex034_scalar_expanded_axes`: a tuple of zero-based axis numbers where size-one values are reused
- `ex034_vector_shape`: a literal Python shape tuple
- `ex034_vector_aligned_shape`: a conceptual shape tuple after left-padding with `1`s for comparison
- `ex034_vector_expanded_axes`: a tuple of zero-based axis numbers where size-one values are reused
- `ex034_compatible`: a Python `bool` stating whether all aligned axes are compatible
- `ex034_out_shape`: a literal Python shape tuple
- `ex034_out`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Scalar expansion over a matrix


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
scalar = torch.tensor(2.0, dtype=DTYPE)
vector = torch.tensor([1.0, 2.0, 3.0], dtype=DTYPE)
_register_case("ex034", {'scalar': scalar, 'vector': vector})


In [ ]:
# Exercise 034: complete only the fields requested above; do not use autograd.
# Define `ex034_scalar_shape`.
# Define `ex034_scalar_aligned_shape`.
# Define `ex034_scalar_expanded_axes`.
# Define `ex034_vector_shape`.
# Define `ex034_vector_aligned_shape`.
# Define `ex034_vector_expanded_axes`.
# Define `ex034_compatible`.
# Define `ex034_out_shape`.
# Define `ex034_out` with PyTorch tensor operations.
# Define `ex034_scalar_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex034_vector_before_op` as an explicit `torch.tensor(...)` literal.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex034_scalar_shape", "ex034", "scalar_shape")
_check_private_value("ex034_scalar_aligned_shape", "ex034", "scalar_aligned_shape")
_check_private_value("ex034_scalar_expanded_axes", "ex034", "scalar_expanded_axes")
_check_private_value("ex034_vector_shape", "ex034", "vector_shape")
_check_private_value("ex034_vector_aligned_shape", "ex034", "vector_aligned_shape")
_check_private_value("ex034_vector_expanded_axes", "ex034", "vector_expanded_axes")
_check_private_value("ex034_compatible", "ex034", "compatible")
_check_private_tensor("ex034_scalar_before_op", "ex034", "scalar_before_op")
_check_private_tensor("ex034_vector_before_op", "ex034", "vector_before_op")
_check_private_value("ex034_out_shape", "ex034", "out_shape")
_check_private_tensor("ex034_out", "ex034", "out")


### Exercise 035 — Scalar expansion over a matrix

**Purpose:** Record two reused result axes for one scalar.

**Visible inputs:** `matrix`, `scalar`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `matrix + scalar`

**Task:** Align both operands, predict the result, then list where each operand expands.

**Ingredients:** Number result axes from left to right starting at zero. `()` means no expansion.

**Axis meaning in this exercise:** `matrix` uses `(rows, columns)`; the scalar has no axes.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.
- **Aligned shape:** a paper-only shape used to show which axes are compared; it does not change the real tensor.
- **Expanded axes:** axis numbers where an aligned size `1` is reused to match the result shape; this is not another shape tuple.

**Explicit operator-ready tensors:** Before calculating the output, write the complete tensor values PyTorch behaves as if each operand has at the common elementwise shape. These are reasoning tensors; PyTorch usually does not allocate these full copies. Define each requested field with a literal `torch.tensor(...)`, not with `expand`, `broadcast_to`, `repeat`, or the original variable. Use the operand's dtype (`DTYPE` for floating tensors and `torch.bool` for Boolean tensors).

**Required predictions and outputs:**

- `ex035_matrix_before_op`: an explicit `torch.tensor(...)` literal showing `matrix` at the common operator-ready shape
- `ex035_scalar_before_op`: an explicit `torch.tensor(...)` literal showing `scalar` at the common operator-ready shape

- `ex035_matrix_shape`: a literal Python shape tuple
- `ex035_matrix_aligned_shape`: a conceptual shape tuple after left-padding with `1`s for comparison
- `ex035_matrix_expanded_axes`: a tuple of zero-based axis numbers where size-one values are reused
- `ex035_scalar_shape`: a literal Python shape tuple
- `ex035_scalar_aligned_shape`: a conceptual shape tuple after left-padding with `1`s for comparison
- `ex035_scalar_expanded_axes`: a tuple of zero-based axis numbers where size-one values are reused
- `ex035_compatible`: a Python `bool` stating whether all aligned axes are compatible
- `ex035_out_shape`: a literal Python shape tuple
- `ex035_out`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Vector expansion down matrix rows


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
matrix = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]], dtype=DTYPE)
scalar = torch.tensor(0.5, dtype=DTYPE)
_register_case("ex035", {'matrix': matrix, 'scalar': scalar})


In [ ]:
# Exercise 035: complete only the fields requested above; do not use autograd.
# Define `ex035_matrix_shape`.
# Define `ex035_matrix_aligned_shape`.
# Define `ex035_matrix_expanded_axes`.
# Define `ex035_scalar_shape`.
# Define `ex035_scalar_aligned_shape`.
# Define `ex035_scalar_expanded_axes`.
# Define `ex035_compatible`.
# Define `ex035_out_shape`.
# Define `ex035_out` with PyTorch tensor operations.
# Define `ex035_matrix_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex035_scalar_before_op` as an explicit `torch.tensor(...)` literal.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex035_matrix_shape", "ex035", "matrix_shape")
_check_private_value("ex035_matrix_aligned_shape", "ex035", "matrix_aligned_shape")
_check_private_value("ex035_matrix_expanded_axes", "ex035", "matrix_expanded_axes")
_check_private_value("ex035_scalar_shape", "ex035", "scalar_shape")
_check_private_value("ex035_scalar_aligned_shape", "ex035", "scalar_aligned_shape")
_check_private_value("ex035_scalar_expanded_axes", "ex035", "scalar_expanded_axes")
_check_private_value("ex035_compatible", "ex035", "compatible")
_check_private_tensor("ex035_matrix_before_op", "ex035", "matrix_before_op")
_check_private_tensor("ex035_scalar_before_op", "ex035", "scalar_before_op")
_check_private_value("ex035_out_shape", "ex035", "out_shape")
_check_private_tensor("ex035_out", "ex035", "out")


### Exercise 036 — Vector expansion down matrix rows

**Purpose:** Connect a conceptual leading one with reuse down rows.

**Visible inputs:** `matrix`, `vector`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `matrix + vector`

**Task:** Write aligned shapes, then identify only the axes whose size-one entries change in the result.

**Ingredients:** Do not put result sizes in an expanded-axes tuple; put axis numbers.

**Axis meaning in this exercise:** `matrix` uses `(rows, columns)`; the vector values belong to columns.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.
- **Aligned shape:** a paper-only shape used to show which axes are compared; it does not change the real tensor.
- **Expanded axes:** axis numbers where an aligned size `1` is reused to match the result shape; this is not another shape tuple.

**Explicit operator-ready tensors:** Before calculating the output, write the complete tensor values PyTorch behaves as if each operand has at the common elementwise shape. These are reasoning tensors; PyTorch usually does not allocate these full copies. Define each requested field with a literal `torch.tensor(...)`, not with `expand`, `broadcast_to`, `repeat`, or the original variable. Use the operand's dtype (`DTYPE` for floating tensors and `torch.bool` for Boolean tensors).

**Required predictions and outputs:**

- `ex036_matrix_before_op`: an explicit `torch.tensor(...)` literal showing `matrix` at the common operator-ready shape
- `ex036_vector_before_op`: an explicit `torch.tensor(...)` literal showing `vector` at the common operator-ready shape

- `ex036_matrix_shape`: a literal Python shape tuple
- `ex036_matrix_aligned_shape`: a conceptual shape tuple after left-padding with `1`s for comparison
- `ex036_matrix_expanded_axes`: a tuple of zero-based axis numbers where size-one values are reused
- `ex036_vector_shape`: a literal Python shape tuple
- `ex036_vector_aligned_shape`: a conceptual shape tuple after left-padding with `1`s for comparison
- `ex036_vector_expanded_axes`: a tuple of zero-based axis numbers where size-one values are reused
- `ex036_compatible`: a Python `bool` stating whether all aligned axes are compatible
- `ex036_out_shape`: a literal Python shape tuple
- `ex036_out`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Explicit row expansion


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
matrix = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]], dtype=DTYPE)
vector = torch.tensor([10.0, 20.0, 30.0], dtype=DTYPE)
_register_case("ex036", {'matrix': matrix, 'vector': vector})


In [ ]:
# Exercise 036: complete only the fields requested above; do not use autograd.
# Define `ex036_matrix_shape`.
# Define `ex036_matrix_aligned_shape`.
# Define `ex036_matrix_expanded_axes`.
# Define `ex036_vector_shape`.
# Define `ex036_vector_aligned_shape`.
# Define `ex036_vector_expanded_axes`.
# Define `ex036_compatible`.
# Define `ex036_out_shape`.
# Define `ex036_out` with PyTorch tensor operations.
# Define `ex036_matrix_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex036_vector_before_op` as an explicit `torch.tensor(...)` literal.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex036_matrix_shape", "ex036", "matrix_shape")
_check_private_value("ex036_matrix_aligned_shape", "ex036", "matrix_aligned_shape")
_check_private_value("ex036_matrix_expanded_axes", "ex036", "matrix_expanded_axes")
_check_private_value("ex036_vector_shape", "ex036", "vector_shape")
_check_private_value("ex036_vector_aligned_shape", "ex036", "vector_aligned_shape")
_check_private_value("ex036_vector_expanded_axes", "ex036", "vector_expanded_axes")
_check_private_value("ex036_compatible", "ex036", "compatible")
_check_private_tensor("ex036_matrix_before_op", "ex036", "matrix_before_op")
_check_private_tensor("ex036_vector_before_op", "ex036", "vector_before_op")
_check_private_value("ex036_out_shape", "ex036", "out_shape")
_check_private_tensor("ex036_out", "ex036", "out")


### Exercise 037 — Explicit row expansion

**Purpose:** See that a real singleton row axis and a conceptual leading one can produce the same reuse pattern.

**Visible inputs:** `matrix`, `row`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `matrix + row`

**Task:** Align, find the result, and record expanded axes.

**Ingredients:** The row already has rank two, so alignment does not add an axis; its existing singleton may still expand.

**Axis meaning in this exercise:** For `matrix` and `row`, axis `0` means rows and axis `1` means columns.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.
- **Aligned shape:** a paper-only shape used to show which axes are compared; it does not change the real tensor.
- **Expanded axes:** axis numbers where an aligned size `1` is reused to match the result shape; this is not another shape tuple.

**Explicit operator-ready tensors:** Before calculating the output, write the complete tensor values PyTorch behaves as if each operand has at the common elementwise shape. These are reasoning tensors; PyTorch usually does not allocate these full copies. Define each requested field with a literal `torch.tensor(...)`, not with `expand`, `broadcast_to`, `repeat`, or the original variable. Use the operand's dtype (`DTYPE` for floating tensors and `torch.bool` for Boolean tensors).

**Required predictions and outputs:**

- `ex037_matrix_before_op`: an explicit `torch.tensor(...)` literal showing `matrix` at the common operator-ready shape
- `ex037_row_before_op`: an explicit `torch.tensor(...)` literal showing `row` at the common operator-ready shape

- `ex037_matrix_shape`: a literal Python shape tuple
- `ex037_matrix_aligned_shape`: a conceptual shape tuple after left-padding with `1`s for comparison
- `ex037_matrix_expanded_axes`: a tuple of zero-based axis numbers where size-one values are reused
- `ex037_row_shape`: a literal Python shape tuple
- `ex037_row_aligned_shape`: a conceptual shape tuple after left-padding with `1`s for comparison
- `ex037_row_expanded_axes`: a tuple of zero-based axis numbers where size-one values are reused
- `ex037_compatible`: a Python `bool` stating whether all aligned axes are compatible
- `ex037_out_shape`: a literal Python shape tuple
- `ex037_out`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Explicit column expansion


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
matrix = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]], dtype=DTYPE)
row = torch.tensor([[10.0, 20.0, 30.0]], dtype=DTYPE)
_register_case("ex037", {'matrix': matrix, 'row': row})


In [ ]:
# Exercise 037: complete only the fields requested above; do not use autograd.
# Define `ex037_matrix_shape`.
# Define `ex037_matrix_aligned_shape`.
# Define `ex037_matrix_expanded_axes`.
# Define `ex037_row_shape`.
# Define `ex037_row_aligned_shape`.
# Define `ex037_row_expanded_axes`.
# Define `ex037_compatible`.
# Define `ex037_out_shape`.
# Define `ex037_out` with PyTorch tensor operations.
# Define `ex037_matrix_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex037_row_before_op` as an explicit `torch.tensor(...)` literal.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex037_matrix_shape", "ex037", "matrix_shape")
_check_private_value("ex037_matrix_aligned_shape", "ex037", "matrix_aligned_shape")
_check_private_value("ex037_matrix_expanded_axes", "ex037", "matrix_expanded_axes")
_check_private_value("ex037_row_shape", "ex037", "row_shape")
_check_private_value("ex037_row_aligned_shape", "ex037", "row_aligned_shape")
_check_private_value("ex037_row_expanded_axes", "ex037", "row_expanded_axes")
_check_private_value("ex037_compatible", "ex037", "compatible")
_check_private_tensor("ex037_matrix_before_op", "ex037", "matrix_before_op")
_check_private_tensor("ex037_row_before_op", "ex037", "row_before_op")
_check_private_value("ex037_out_shape", "ex037", "out_shape")
_check_private_tensor("ex037_out", "ex037", "out")


### Exercise 038 — Explicit column expansion

**Purpose:** Identify reuse across columns rather than rows.

**Visible inputs:** `matrix`, `column`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `matrix + column`

**Task:** Align, determine output shape, and list the column operand's expanded axis.

**Ingredients:** Axis `0` is rows and axis `1` is columns.

**Axis meaning in this exercise:** For `matrix` and `column`, axis `0` means rows and axis `1` means columns.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.
- **Aligned shape:** a paper-only shape used to show which axes are compared; it does not change the real tensor.
- **Expanded axes:** axis numbers where an aligned size `1` is reused to match the result shape; this is not another shape tuple.

**Explicit operator-ready tensors:** Before calculating the output, write the complete tensor values PyTorch behaves as if each operand has at the common elementwise shape. These are reasoning tensors; PyTorch usually does not allocate these full copies. Define each requested field with a literal `torch.tensor(...)`, not with `expand`, `broadcast_to`, `repeat`, or the original variable. Use the operand's dtype (`DTYPE` for floating tensors and `torch.bool` for Boolean tensors).

**Required predictions and outputs:**

- `ex038_matrix_before_op`: an explicit `torch.tensor(...)` literal showing `matrix` at the common operator-ready shape
- `ex038_column_before_op`: an explicit `torch.tensor(...)` literal showing `column` at the common operator-ready shape

- `ex038_matrix_shape`: a literal Python shape tuple
- `ex038_matrix_aligned_shape`: a conceptual shape tuple after left-padding with `1`s for comparison
- `ex038_matrix_expanded_axes`: a tuple of zero-based axis numbers where size-one values are reused
- `ex038_column_shape`: a literal Python shape tuple
- `ex038_column_aligned_shape`: a conceptual shape tuple after left-padding with `1`s for comparison
- `ex038_column_expanded_axes`: a tuple of zero-based axis numbers where size-one values are reused
- `ex038_compatible`: a Python `bool` stating whether all aligned axes are compatible
- `ex038_out_shape`: a literal Python shape tuple
- `ex038_out`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Two operands expand


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
matrix = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]], dtype=DTYPE)
column = torch.tensor([[10.0], [20.0]], dtype=DTYPE)
_register_case("ex038", {'matrix': matrix, 'column': column})


In [ ]:
# Exercise 038: complete only the fields requested above; do not use autograd.
# Define `ex038_matrix_shape`.
# Define `ex038_matrix_aligned_shape`.
# Define `ex038_matrix_expanded_axes`.
# Define `ex038_column_shape`.
# Define `ex038_column_aligned_shape`.
# Define `ex038_column_expanded_axes`.
# Define `ex038_compatible`.
# Define `ex038_out_shape`.
# Define `ex038_out` with PyTorch tensor operations.
# Define `ex038_matrix_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex038_column_before_op` as an explicit `torch.tensor(...)` literal.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex038_matrix_shape", "ex038", "matrix_shape")
_check_private_value("ex038_matrix_aligned_shape", "ex038", "matrix_aligned_shape")
_check_private_value("ex038_matrix_expanded_axes", "ex038", "matrix_expanded_axes")
_check_private_value("ex038_column_shape", "ex038", "column_shape")
_check_private_value("ex038_column_aligned_shape", "ex038", "column_aligned_shape")
_check_private_value("ex038_column_expanded_axes", "ex038", "column_expanded_axes")
_check_private_value("ex038_compatible", "ex038", "compatible")
_check_private_tensor("ex038_matrix_before_op", "ex038", "matrix_before_op")
_check_private_tensor("ex038_column_before_op", "ex038", "column_before_op")
_check_private_value("ex038_out_shape", "ex038", "out_shape")
_check_private_tensor("ex038_out", "ex038", "out")


### Exercise 039 — Two operands expand

**Purpose:** Track separate reuse axes for a column and row.

**Visible inputs:** `column`, `row`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `column + row`

**Task:** Align first, then compare each operand independently with the result.

**Ingredients:** An expanded-axes tuple belongs to one operand, not to the operation as a whole.

**Axis meaning in this exercise:** For both tensors, axis `0` represents rows and axis `1` represents columns.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.
- **Aligned shape:** a paper-only shape used to show which axes are compared; it does not change the real tensor.
- **Expanded axes:** axis numbers where an aligned size `1` is reused to match the result shape; this is not another shape tuple.

**Explicit operator-ready tensors:** Before calculating the output, write the complete tensor values PyTorch behaves as if each operand has at the common elementwise shape. These are reasoning tensors; PyTorch usually does not allocate these full copies. Define each requested field with a literal `torch.tensor(...)`, not with `expand`, `broadcast_to`, `repeat`, or the original variable. Use the operand's dtype (`DTYPE` for floating tensors and `torch.bool` for Boolean tensors).

**Required predictions and outputs:**

- `ex039_column_before_op`: an explicit `torch.tensor(...)` literal showing `column` at the common operator-ready shape
- `ex039_row_before_op`: an explicit `torch.tensor(...)` literal showing `row` at the common operator-ready shape

- `ex039_column_shape`: a literal Python shape tuple
- `ex039_column_aligned_shape`: a conceptual shape tuple after left-padding with `1`s for comparison
- `ex039_column_expanded_axes`: a tuple of zero-based axis numbers where size-one values are reused
- `ex039_row_shape`: a literal Python shape tuple
- `ex039_row_aligned_shape`: a conceptual shape tuple after left-padding with `1`s for comparison
- `ex039_row_expanded_axes`: a tuple of zero-based axis numbers where size-one values are reused
- `ex039_compatible`: a Python `bool` stating whether all aligned axes are compatible
- `ex039_out_shape`: a literal Python shape tuple
- `ex039_out`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Feature vector expansion across three axes


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
column = torch.tensor([[1.0], [2.0]], dtype=DTYPE)
row = torch.tensor([[10.0, 20.0, 30.0]], dtype=DTYPE)
_register_case("ex039", {'column': column, 'row': row})


In [ ]:
# Exercise 039: complete only the fields requested above; do not use autograd.
# Define `ex039_column_shape`.
# Define `ex039_column_aligned_shape`.
# Define `ex039_column_expanded_axes`.
# Define `ex039_row_shape`.
# Define `ex039_row_aligned_shape`.
# Define `ex039_row_expanded_axes`.
# Define `ex039_compatible`.
# Define `ex039_out_shape`.
# Define `ex039_out` with PyTorch tensor operations.
# Define `ex039_column_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex039_row_before_op` as an explicit `torch.tensor(...)` literal.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex039_column_shape", "ex039", "column_shape")
_check_private_value("ex039_column_aligned_shape", "ex039", "column_aligned_shape")
_check_private_value("ex039_column_expanded_axes", "ex039", "column_expanded_axes")
_check_private_value("ex039_row_shape", "ex039", "row_shape")
_check_private_value("ex039_row_aligned_shape", "ex039", "row_aligned_shape")
_check_private_value("ex039_row_expanded_axes", "ex039", "row_expanded_axes")
_check_private_value("ex039_compatible", "ex039", "compatible")
_check_private_tensor("ex039_column_before_op", "ex039", "column_before_op")
_check_private_tensor("ex039_row_before_op", "ex039", "row_before_op")
_check_private_value("ex039_out_shape", "ex039", "out_shape")
_check_private_tensor("ex039_out", "ex039", "out")


### Exercise 040 — Feature vector expansion across three axes

**Purpose:** Identify reuse over batch and time after right alignment.

**Visible inputs:** `tokens`, `features`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `tokens * features`

**Task:** Align the feature vector, find the result, then list its reused result axes.

**Ingredients:** Axes are `(batch, time, features)`. Expanded axes describe reuse; they are mainly a diagnostic tool.

**Axis meaning in this exercise:** `tokens` uses `(batch, time, features)`; the vector values belong to features.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.
- **Aligned shape:** a paper-only shape used to show which axes are compared; it does not change the real tensor.
- **Expanded axes:** axis numbers where an aligned size `1` is reused to match the result shape; this is not another shape tuple.

**Explicit operator-ready tensors:** Before calculating the output, write the complete tensor values PyTorch behaves as if each operand has at the common elementwise shape. These are reasoning tensors; PyTorch usually does not allocate these full copies. Define each requested field with a literal `torch.tensor(...)`, not with `expand`, `broadcast_to`, `repeat`, or the original variable. Use the operand's dtype (`DTYPE` for floating tensors and `torch.bool` for Boolean tensors).

**Required predictions and outputs:**

- `ex040_tokens_before_op`: an explicit `torch.tensor(...)` literal showing `tokens` at the common operator-ready shape
- `ex040_features_before_op`: an explicit `torch.tensor(...)` literal showing `features` at the common operator-ready shape

- `ex040_tokens_shape`: a literal Python shape tuple
- `ex040_tokens_aligned_shape`: a conceptual shape tuple after left-padding with `1`s for comparison
- `ex040_tokens_expanded_axes`: a tuple of zero-based axis numbers where size-one values are reused
- `ex040_features_shape`: a literal Python shape tuple
- `ex040_features_aligned_shape`: a conceptual shape tuple after left-padding with `1`s for comparison
- `ex040_features_expanded_axes`: a tuple of zero-based axis numbers where size-one values are reused
- `ex040_compatible`: a Python `bool` stating whether all aligned axes are compatible
- `ex040_out_shape`: a literal Python shape tuple
- `ex040_out`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Length two does not mean matrix rows


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
tokens = torch.arange(24, dtype=DTYPE).reshape(2, 3, 4)
features = torch.tensor([1.0, 2.0, 3.0, 4.0], dtype=DTYPE)
_register_case("ex040", {'tokens': tokens, 'features': features})


In [ ]:
# Exercise 040: complete only the fields requested above; do not use autograd.
# Define `ex040_tokens_shape`.
# Define `ex040_tokens_aligned_shape`.
# Define `ex040_tokens_expanded_axes`.
# Define `ex040_features_shape`.
# Define `ex040_features_aligned_shape`.
# Define `ex040_features_expanded_axes`.
# Define `ex040_compatible`.
# Define `ex040_out_shape`.
# Define `ex040_out` with PyTorch tensor operations.
# Define `ex040_tokens_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex040_features_before_op` as an explicit `torch.tensor(...)` literal.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex040_tokens_shape", "ex040", "tokens_shape")
_check_private_value("ex040_tokens_aligned_shape", "ex040", "tokens_aligned_shape")
_check_private_value("ex040_tokens_expanded_axes", "ex040", "tokens_expanded_axes")
_check_private_value("ex040_features_shape", "ex040", "features_shape")
_check_private_value("ex040_features_aligned_shape", "ex040", "features_aligned_shape")
_check_private_value("ex040_features_expanded_axes", "ex040", "features_expanded_axes")
_check_private_value("ex040_compatible", "ex040", "compatible")
_check_private_tensor("ex040_tokens_before_op", "ex040", "tokens_before_op")
_check_private_tensor("ex040_features_before_op", "ex040", "features_before_op")
_check_private_value("ex040_out_shape", "ex040", "out_shape")
_check_private_tensor("ex040_out", "ex040", "out")


## 6. Invalid shapes and deliberate repairs

Diagnose incompatible axes and add explicit size-one axes so the shape clearly expresses what each value belongs to.

This section covers Exercises 041–048.


### Exercise 041 — Length two does not mean matrix rows

**Purpose:** See why PyTorch follows shape positions rather than guessing that two values were meant for two rows.

**Visible inputs:** `values`, `matrix`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `values + matrix`

**Task:** Align the shapes and identify every incompatible axis without executing the invalid addition yourself.

**Ingredients:** The vector aligns with the final matrix axis, not automatically with rows.

**Axis meaning in this exercise:** `matrix` uses `(rows, columns)`. The two values were intended for rows, but vector axis `0` right-aligns with columns.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.
- **Aligned shape:** a paper-only shape used to show which axes are compared; it does not change the real tensor.

**Required predictions and outputs:**

- `ex041_values_shape`: a literal Python shape tuple
- `ex041_values_aligned_shape`: a conceptual shape tuple after left-padding with `1`s for comparison
- `ex041_matrix_shape`: a literal Python shape tuple
- `ex041_matrix_aligned_shape`: a conceptual shape tuple after left-padding with `1`s for comparison
- `ex041_compatible`: a Python `bool` stating whether all aligned axes are compatible
- `ex041_incompatible_axes`: a tuple of zero-based aligned axes with unequal non-one sizes

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Transposed matrix shapes


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
values = torch.tensor([10.0, 20.0], dtype=DTYPE)
matrix = torch.ones((2, 3), dtype=DTYPE)
_register_case("ex041", {'values': values, 'matrix': matrix})


In [ ]:
# Exercise 041: complete only the fields requested above; do not use autograd.
# Define `ex041_values_shape`.
# Define `ex041_values_aligned_shape`.
# Define `ex041_matrix_shape`.
# Define `ex041_matrix_aligned_shape`.
# Define `ex041_compatible`.
# Define `ex041_incompatible_axes`.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex041_values_shape", "ex041", "values_shape")
_check_private_value("ex041_values_aligned_shape", "ex041", "values_aligned_shape")
_check_private_value("ex041_matrix_shape", "ex041", "matrix_shape")
_check_private_value("ex041_matrix_aligned_shape", "ex041", "matrix_aligned_shape")
_check_private_value("ex041_compatible", "ex041", "compatible")
_check_private_value("ex041_incompatible_axes", "ex041", "incompatible_axes")


### Exercise 042 — Transposed matrix shapes

**Purpose:** See that equal element counts do not imply elementwise compatibility.

**Visible inputs:** `a`, `b`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `a + b`

**Task:** Align both shapes and identify mismatching axes.

**Ingredients:** Elementwise compatibility compares each axis size; it does not flatten tensors.

**Axis meaning in this exercise:** For both matrices, axis `0` means rows and axis `1` means columns.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.
- **Aligned shape:** a paper-only shape used to show which axes are compared; it does not change the real tensor.

**Required predictions and outputs:**

- `ex042_a_shape`: a literal Python shape tuple
- `ex042_a_aligned_shape`: a conceptual shape tuple after left-padding with `1`s for comparison
- `ex042_b_shape`: a literal Python shape tuple
- `ex042_b_aligned_shape`: a conceptual shape tuple after left-padding with `1`s for comparison
- `ex042_compatible`: a Python `bool` stating whether all aligned axes are compatible
- `ex042_incompatible_axes`: a tuple of zero-based aligned axes with unequal non-one sizes

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Wrong feature-vector length


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
a = torch.ones((2, 3), dtype=DTYPE)
b = torch.ones((3, 2), dtype=DTYPE)
_register_case("ex042", {'a': a, 'b': b})


In [ ]:
# Exercise 042: complete only the fields requested above; do not use autograd.
# Define `ex042_a_shape`.
# Define `ex042_a_aligned_shape`.
# Define `ex042_b_shape`.
# Define `ex042_b_aligned_shape`.
# Define `ex042_compatible`.
# Define `ex042_incompatible_axes`.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex042_a_shape", "ex042", "a_shape")
_check_private_value("ex042_a_aligned_shape", "ex042", "a_aligned_shape")
_check_private_value("ex042_b_shape", "ex042", "b_shape")
_check_private_value("ex042_b_aligned_shape", "ex042", "b_aligned_shape")
_check_private_value("ex042_compatible", "ex042", "compatible")
_check_private_value("ex042_incompatible_axes", "ex042", "incompatible_axes")


### Exercise 043 — Wrong feature-vector length

**Purpose:** Locate a mismatch on the final rank-three axis.

**Visible inputs:** `tokens`, `features`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `tokens + features`

**Task:** Right-align and identify the incompatible axis.

**Ingredients:** A rank-one tensor always begins under the rightmost axis during comparison.

**Axis meaning in this exercise:** `tokens` uses `(batch, time, features)`; the vector is intended to supply feature values.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.
- **Aligned shape:** a paper-only shape used to show which axes are compared; it does not change the real tensor.

**Required predictions and outputs:**

- `ex043_tokens_shape`: a literal Python shape tuple
- `ex043_tokens_aligned_shape`: a conceptual shape tuple after left-padding with `1`s for comparison
- `ex043_features_shape`: a literal Python shape tuple
- `ex043_features_aligned_shape`: a conceptual shape tuple after left-padding with `1`s for comparison
- `ex043_compatible`: a Python `bool` stating whether all aligned axes are compatible
- `ex043_incompatible_axes`: a tuple of zero-based aligned axes with unequal non-one sizes

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Missing middle singleton


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
tokens = torch.ones((2, 3, 4), dtype=DTYPE)
features = torch.tensor([1.0, 2.0, 3.0], dtype=DTYPE)
_register_case("ex043", {'tokens': tokens, 'features': features})


In [ ]:
# Exercise 043: complete only the fields requested above; do not use autograd.
# Define `ex043_tokens_shape`.
# Define `ex043_tokens_aligned_shape`.
# Define `ex043_features_shape`.
# Define `ex043_features_aligned_shape`.
# Define `ex043_compatible`.
# Define `ex043_incompatible_axes`.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex043_tokens_shape", "ex043", "tokens_shape")
_check_private_value("ex043_tokens_aligned_shape", "ex043", "tokens_aligned_shape")
_check_private_value("ex043_features_shape", "ex043", "features_shape")
_check_private_value("ex043_features_aligned_shape", "ex043", "features_aligned_shape")
_check_private_value("ex043_compatible", "ex043", "compatible")
_check_private_value("ex043_incompatible_axes", "ex043", "incompatible_axes")


### Exercise 044 — Missing middle singleton

**Purpose:** Diagnose a `(batch, features)` tensor that cannot directly represent `(batch, time, features)` values.

**Visible inputs:** `sequence`, `per_batch_feature`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `sequence + per_batch_feature`

**Task:** Align the operands and identify the mismatching axis.

**Ingredients:** Right alignment treats `(2, 4)` as conceptual `(1, 2, 4)`, not `(2, 1, 4)`.

**Axis meaning in this exercise:** `sequence` uses `(batch, time, features)`; `per_batch_feature` uses `(batch, features)`.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.
- **Aligned shape:** a paper-only shape used to show which axes are compared; it does not change the real tensor.

**Required predictions and outputs:**

- `ex044_sequence_shape`: a literal Python shape tuple
- `ex044_sequence_aligned_shape`: a conceptual shape tuple after left-padding with `1`s for comparison
- `ex044_per_batch_feature_shape`: a literal Python shape tuple
- `ex044_per_batch_feature_aligned_shape`: a conceptual shape tuple after left-padding with `1`s for comparison
- `ex044_compatible`: a Python `bool` stating whether all aligned axes are compatible
- `ex044_incompatible_axes`: a tuple of zero-based aligned axes with unequal non-one sizes

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Unprepared image channels


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
sequence = torch.ones((2, 3, 4), dtype=DTYPE)
per_batch_feature = torch.ones((2, 4), dtype=DTYPE)
_register_case("ex044", {'sequence': sequence, 'per_batch_feature': per_batch_feature})


In [ ]:
# Exercise 044: complete only the fields requested above; do not use autograd.
# Define `ex044_sequence_shape`.
# Define `ex044_sequence_aligned_shape`.
# Define `ex044_per_batch_feature_shape`.
# Define `ex044_per_batch_feature_aligned_shape`.
# Define `ex044_compatible`.
# Define `ex044_incompatible_axes`.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex044_sequence_shape", "ex044", "sequence_shape")
_check_private_value("ex044_sequence_aligned_shape", "ex044", "sequence_aligned_shape")
_check_private_value("ex044_per_batch_feature_shape", "ex044", "per_batch_feature_shape")
_check_private_value("ex044_per_batch_feature_aligned_shape", "ex044", "per_batch_feature_aligned_shape")
_check_private_value("ex044_compatible", "ex044", "compatible")
_check_private_value("ex044_incompatible_axes", "ex044", "incompatible_axes")


### Exercise 045 — Unprepared image channels

**Purpose:** See why a channel vector does not naturally align with the `(batch, channel, height, width)` image layout.

**Visible inputs:** `images`, `channels`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `images + channels`

**Task:** Right-align and identify the incompatible final axis.

**Ingredients:** Image axes are `(batch, channel, height, width)`. A plain vector aligns with width, not channel.

**Axis meaning in this exercise:** `images` uses `(batch, channel, height, width)`; the vector values belong to channels.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.
- **Aligned shape:** a paper-only shape used to show which axes are compared; it does not change the real tensor.

**Required predictions and outputs:**

- `ex045_images_shape`: a literal Python shape tuple
- `ex045_images_aligned_shape`: a conceptual shape tuple after left-padding with `1`s for comparison
- `ex045_channels_shape`: a literal Python shape tuple
- `ex045_channels_aligned_shape`: a conceptual shape tuple after left-padding with `1`s for comparison
- `ex045_compatible`: a Python `bool` stating whether all aligned axes are compatible
- `ex045_incompatible_axes`: a tuple of zero-based aligned axes with unequal non-one sizes

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Repair row-owned values


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
images = torch.ones((2, 3, 4, 5), dtype=DTYPE)
channels = torch.tensor([0.1, 0.2, 0.3], dtype=DTYPE)
_register_case("ex045", {'images': images, 'channels': channels})


In [ ]:
# Exercise 045: complete only the fields requested above; do not use autograd.
# Define `ex045_images_shape`.
# Define `ex045_images_aligned_shape`.
# Define `ex045_channels_shape`.
# Define `ex045_channels_aligned_shape`.
# Define `ex045_compatible`.
# Define `ex045_incompatible_axes`.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex045_images_shape", "ex045", "images_shape")
_check_private_value("ex045_images_aligned_shape", "ex045", "images_aligned_shape")
_check_private_value("ex045_channels_shape", "ex045", "channels_shape")
_check_private_value("ex045_channels_aligned_shape", "ex045", "channels_aligned_shape")
_check_private_value("ex045_compatible", "ex045", "compatible")
_check_private_value("ex045_incompatible_axes", "ex045", "incompatible_axes")


### Exercise 046 — Repair row-owned values

**Purpose:** Turn an invalid row-intended vector into an explicit column shape.

**Visible inputs:** `values`, `matrix`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `prepared + matrix`

**Task:** Create `prepared = values[:, None]`, then add one value per row.

**Ingredients:** The inserted trailing singleton makes semantic row ownership visible.

**Axis meaning in this exercise:** `matrix` uses `(rows, columns)`; axis `0` of `values` is intended to represent rows.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.
- **Aligned shape:** a paper-only shape used to show which axes are compared; it does not change the real tensor.

**Required predictions and outputs:**

- `ex046_values_shape`: a literal Python shape tuple
- `ex046_matrix_shape`: a literal Python shape tuple
- `ex046_prepared_shape`: a literal Python shape tuple
- `ex046_prepared`: the requested PyTorch tensor
- `ex046_out_shape`: a literal Python shape tuple
- `ex046_out`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Repair per-batch features


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
values = torch.tensor([10.0, 20.0], dtype=DTYPE)
matrix = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]], dtype=DTYPE)
_register_case("ex046", {'values': values, 'matrix': matrix})


In [ ]:
# Exercise 046: complete only the fields requested above; do not use autograd.
# Define `ex046_values_shape`.
# Define `ex046_matrix_shape`.
# Define `ex046_prepared_shape`.
# Define `ex046_prepared` with PyTorch tensor operations.
# Define `ex046_out_shape`.
# Define `ex046_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex046_values_shape", "ex046", "values_shape")
_check_private_value("ex046_matrix_shape", "ex046", "matrix_shape")
_check_private_value("ex046_prepared_shape", "ex046", "prepared_shape")
_check_private_tensor("ex046_prepared", "ex046", "prepared")
_check_private_value("ex046_out_shape", "ex046", "out_shape")
_check_private_tensor("ex046_out", "ex046", "out")


### Exercise 047 — Repair per-batch features

**Purpose:** Insert a missing time axis deliberately.

**Visible inputs:** `sequence`, `per_batch_feature`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `prepared + sequence`

**Task:** Create `prepared = per_batch_feature[:, None, :]`, then add it across time.

**Ingredients:** Axes become `(batch, 1, features)` before broadcasting with `(batch, time, features)`.

**Axis meaning in this exercise:** `sequence` uses `(batch, time, features)`; `per_batch_feature` uses `(batch, features)`.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.
- **Aligned shape:** a paper-only shape used to show which axes are compared; it does not change the real tensor.

**Required predictions and outputs:**

- `ex047_sequence_shape`: a literal Python shape tuple
- `ex047_per_batch_feature_shape`: a literal Python shape tuple
- `ex047_prepared_shape`: a literal Python shape tuple
- `ex047_prepared`: the requested PyTorch tensor
- `ex047_out_shape`: a literal Python shape tuple
- `ex047_out`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Three-way where mismatch


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
sequence = torch.arange(24, dtype=DTYPE).reshape(2, 3, 4)
per_batch_feature = torch.tensor([[1.0, 2.0, 3.0, 4.0], [5.0, 6.0, 7.0, 8.0]], dtype=DTYPE)
_register_case("ex047", {'sequence': sequence, 'per_batch_feature': per_batch_feature})


In [ ]:
# Exercise 047: complete only the fields requested above; do not use autograd.
# Define `ex047_sequence_shape`.
# Define `ex047_per_batch_feature_shape`.
# Define `ex047_prepared_shape`.
# Define `ex047_prepared` with PyTorch tensor operations.
# Define `ex047_out_shape`.
# Define `ex047_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex047_sequence_shape", "ex047", "sequence_shape")
_check_private_value("ex047_per_batch_feature_shape", "ex047", "per_batch_feature_shape")
_check_private_value("ex047_prepared_shape", "ex047", "prepared_shape")
_check_private_tensor("ex047_prepared", "ex047", "prepared")
_check_private_value("ex047_out_shape", "ex047", "out_shape")
_check_private_tensor("ex047_out", "ex047", "out")


### Exercise 048 — Three-way where mismatch

**Purpose:** Check the shapes used by `torch.where`: the Boolean mask, values chosen where it is true, and values chosen where it is false.

**Visible inputs:** `mask`, `x`, `y`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `torch.where(mask, x, y)`

**Task:** Align all three operands and identify incompatible axes.

**Ingredients:** All participating tensors must broadcast to one common result shape.

**Axis meaning in this exercise:** Treat every tensor as a row-column grid: axis `0` is rows and axis `1` is columns.

**New operation or idea explained:** `torch.where(mask, x, y)` chooses a value from `x` wherever the Boolean `mask` is `True`, and from `y` wherever it is `False`. The mask, `x`, and `y` must all broadcast to one common shape.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.
- **Aligned shape:** a paper-only shape used to show which axes are compared; it does not change the real tensor.

**Required predictions and outputs:**

- `ex048_mask_shape`: a literal Python shape tuple
- `ex048_mask_aligned_shape`: a conceptual shape tuple after left-padding with `1`s for comparison
- `ex048_x_shape`: a literal Python shape tuple
- `ex048_x_aligned_shape`: a conceptual shape tuple after left-padding with `1`s for comparison
- `ex048_y_shape`: a literal Python shape tuple
- `ex048_y_aligned_shape`: a conceptual shape tuple after left-padding with `1`s for comparison
- `ex048_compatible`: a Python `bool` stating whether all aligned axes are compatible
- `ex048_incompatible_axes`: a tuple of zero-based aligned axes with unequal non-one sizes

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Batch feature bias


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
mask = torch.ones((2, 3), dtype=torch.bool)
x = torch.ones((2, 1), dtype=DTYPE)
y = torch.ones((4, 3), dtype=DTYPE)
_register_case("ex048", {'mask': mask, 'x': x, 'y': y})


In [ ]:
# Exercise 048: complete only the fields requested above; do not use autograd.
# Define `ex048_mask_shape`.
# Define `ex048_mask_aligned_shape`.
# Define `ex048_x_shape`.
# Define `ex048_x_aligned_shape`.
# Define `ex048_y_shape`.
# Define `ex048_y_aligned_shape`.
# Define `ex048_compatible`.
# Define `ex048_incompatible_axes`.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex048_mask_shape", "ex048", "mask_shape")
_check_private_value("ex048_mask_aligned_shape", "ex048", "mask_aligned_shape")
_check_private_value("ex048_x_shape", "ex048", "x_shape")
_check_private_value("ex048_x_aligned_shape", "ex048", "x_aligned_shape")
_check_private_value("ex048_y_shape", "ex048", "y_shape")
_check_private_value("ex048_y_aligned_shape", "ex048", "y_aligned_shape")
_check_private_value("ex048_compatible", "ex048", "compatible")
_check_private_value("ex048_incompatible_axes", "ex048", "incompatible_axes")


## 7. Machine-learning axis patterns

Apply broadcasting to examples, features, tokens, image channels, attention axes, and pairwise items.

This section covers Exercises 049–056.


### Exercise 049 — Batch feature bias

**Purpose:** Recognize the standard `(examples, features) + (features,)` pattern.

**Visible inputs:** `activations`, `bias`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `activations + bias`

**Task:** Add one bias per feature to every example.

**Ingredients:** Axes are `(examples, features)`; the feature vector aligns last.

**Axis meaning in this exercise:** `activations` uses `(examples, features)`; the bias vector values belong to features.

**New operation or idea explained:** A **bias** is a value added to another value. Here there is one bias value for each feature, and the same feature biases are used for every example.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Required predictions and outputs:**

- `ex049_activations_shape`: a literal Python shape tuple
- `ex049_bias_shape`: a literal Python shape tuple
- `ex049_out_shape`: a literal Python shape tuple
- `ex049_out`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Per-example scalar offset


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
activations = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0]], dtype=DTYPE)
bias = torch.tensor([0.1, 0.2, 0.3], dtype=DTYPE)
_register_case("ex049", {'activations': activations, 'bias': bias})


In [ ]:
# Exercise 049: complete only the fields requested above; do not use autograd.
# Define `ex049_activations_shape`.
# Define `ex049_bias_shape`.
# Define `ex049_out_shape`.
# Define `ex049_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex049_activations_shape", "ex049", "activations_shape")
_check_private_value("ex049_bias_shape", "ex049", "bias_shape")
_check_private_value("ex049_out_shape", "ex049", "out_shape")
_check_private_tensor("ex049_out", "ex049", "out")


### Exercise 050 — Per-example scalar offset

**Purpose:** Prepare one value for every feature in each example.

**Visible inputs:** `activations`, `offset`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `offset_view + activations`

**Task:** Create `offset_view = offset[:, None]`, then add it across features.

**Ingredients:** The prepared axes are `(examples, 1)`.

**Axis meaning in this exercise:** `activations` uses `(examples, features)`; offset axis `0` identifies examples.

**New operation or idea explained:** An **offset** is an added value. Here each example owns one offset that must be reused across all of that example’s features.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Required predictions and outputs:**

- `ex050_activations_shape`: a literal Python shape tuple
- `ex050_offset_shape`: a literal Python shape tuple
- `ex050_offset_view_shape`: a literal Python shape tuple
- `ex050_offset_view`: the requested PyTorch tensor
- `ex050_out_shape`: a literal Python shape tuple
- `ex050_out`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Position embeddings across batches


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
activations = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]], dtype=DTYPE)
offset = torch.tensor([0.5, -1.0], dtype=DTYPE)
_register_case("ex050", {'activations': activations, 'offset': offset})


In [ ]:
# Exercise 050: complete only the fields requested above; do not use autograd.
# Define `ex050_activations_shape`.
# Define `ex050_offset_shape`.
# Define `ex050_offset_view_shape`.
# Define `ex050_offset_view` with PyTorch tensor operations.
# Define `ex050_out_shape`.
# Define `ex050_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex050_activations_shape", "ex050", "activations_shape")
_check_private_value("ex050_offset_shape", "ex050", "offset_shape")
_check_private_value("ex050_offset_view_shape", "ex050", "offset_view_shape")
_check_private_tensor("ex050_offset_view", "ex050", "offset_view")
_check_private_value("ex050_out_shape", "ex050", "out_shape")
_check_private_tensor("ex050_out", "ex050", "out")


### Exercise 051 — Position embeddings across batches

**Purpose:** Reuse one `(time, features)` table for every batch item.

**Visible inputs:** `tokens`, `positions`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `tokens + positions`

**Task:** Add one position-feature row to each batch.

**Ingredients:** Axes are `(batch, time, features)` and `(time, features)`.

**Axis meaning in this exercise:** `tokens` uses `(batch, time, features)`; `positions` uses `(time, features)`.

**New operation or idea explained:** A **position embedding** is a row of feature values associated with one sequence position. The same position table is added to every batch item.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Required predictions and outputs:**

- `ex051_tokens_shape`: a literal Python shape tuple
- `ex051_positions_shape`: a literal Python shape tuple
- `ex051_out_shape`: a literal Python shape tuple
- `ex051_out`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Token validity mask


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
tokens = torch.arange(24, dtype=DTYPE).reshape(2, 3, 4)
positions = torch.tensor([[-0.5, 0.0, 0.5, 1.0], [0.0, 0.5, 1.0, -0.5], [0.5, 1.0, -0.5, 0.0]], dtype=DTYPE)
_register_case("ex051", {'tokens': tokens, 'positions': positions})


In [ ]:
# Exercise 051: complete only the fields requested above; do not use autograd.
# Define `ex051_tokens_shape`.
# Define `ex051_positions_shape`.
# Define `ex051_out_shape`.
# Define `ex051_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex051_tokens_shape", "ex051", "tokens_shape")
_check_private_value("ex051_positions_shape", "ex051", "positions_shape")
_check_private_value("ex051_out_shape", "ex051", "out_shape")
_check_private_tensor("ex051_out", "ex051", "out")


### Exercise 052 — Token validity mask

**Purpose:** Expand one validity value over all embedding features.

**Visible inputs:** `mask`, `embeddings`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `mask_view * embeddings`

**Task:** Create `mask_view = mask[:, :, None]`, then zero all features at invalid token positions.

**Ingredients:** Mask axes are `(batch, time)`; insert a singleton feature axis.

**Axis meaning in this exercise:** `mask` uses `(batch, time)`; `embeddings` uses `(batch, time, features)`.

**New operation or idea explained:** A **mask** marks which positions to keep. Here `1.0` means keep the embedding and `0.0` means replace it with zeros through multiplication. An **embedding** is the feature vector representing one token.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Explicit operator-ready tensors:** Before calculating the output, write the complete tensor values PyTorch behaves as if each operand has at the common elementwise shape. These are reasoning tensors; PyTorch usually does not allocate these full copies. Define each requested field with a literal `torch.tensor(...)`, not with `expand`, `broadcast_to`, `repeat`, or the original variable. Use the operand's dtype (`DTYPE` for floating tensors and `torch.bool` for Boolean tensors).

**Required predictions and outputs:**

- `ex052_mask_view_before_op`: an explicit `torch.tensor(...)` literal showing `mask_view` at the common operator-ready shape
- `ex052_embeddings_before_op`: an explicit `torch.tensor(...)` literal showing `embeddings` at the common operator-ready shape

- `ex052_mask_shape`: a literal Python shape tuple
- `ex052_embeddings_shape`: a literal Python shape tuple
- `ex052_mask_view_shape`: a literal Python shape tuple
- `ex052_mask_view`: the requested PyTorch tensor
- `ex052_out_shape`: a literal Python shape tuple
- `ex052_out`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Class weights


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
mask = torch.tensor([[1.0, 1.0, 0.0], [1.0, 0.0, 0.0]], dtype=DTYPE)
embeddings = torch.arange(24, dtype=DTYPE).reshape(2, 3, 4)
_register_case("ex052", {'mask': mask, 'embeddings': embeddings})


In [ ]:
# Exercise 052: complete only the fields requested above; do not use autograd.
# Define `ex052_mask_shape`.
# Define `ex052_embeddings_shape`.
# Define `ex052_mask_view_shape`.
# Define `ex052_mask_view` with PyTorch tensor operations.
# Define `ex052_out_shape`.
# Define `ex052_out` with PyTorch tensor operations.
# Define `ex052_mask_view_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex052_embeddings_before_op` as an explicit `torch.tensor(...)` literal.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex052_mask_shape", "ex052", "mask_shape")
_check_private_value("ex052_embeddings_shape", "ex052", "embeddings_shape")
_check_private_value("ex052_mask_view_shape", "ex052", "mask_view_shape")
_check_private_tensor("ex052_mask_view", "ex052", "mask_view")
_check_private_tensor("ex052_mask_view_before_op", "ex052", "mask_view_before_op")
_check_private_tensor("ex052_embeddings_before_op", "ex052", "embeddings_before_op")
_check_private_value("ex052_out_shape", "ex052", "out_shape")
_check_private_tensor("ex052_out", "ex052", "out")


### Exercise 053 — Class weights

**Purpose:** Apply one weight per candidate class to every example.

**Visible inputs:** `losses`, `class_weights`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `losses * class_weights`

**Task:** Weight every candidate-class loss.

**Ingredients:** Axes are `(examples, classes)` and `(classes,)`. This weights all candidate losses; no target indexing occurs here.

**Axis meaning in this exercise:** `losses` uses `(examples, classes)`; the weight vector values belong to classes.

**New operation or idea explained:** A **loss** is a number measuring error. A **class** is one candidate category. This exercise multiplies every candidate-class loss by the weight assigned to that class; it does not choose a target class.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Required predictions and outputs:**

- `ex053_losses_shape`: a literal Python shape tuple
- `ex053_class_weights_shape`: a literal Python shape tuple
- `ex053_out_shape`: a literal Python shape tuple
- `ex053_out`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Image channel bias


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
losses = torch.tensor([[0.2, 0.5, 1.0], [0.3, 0.4, 0.8]], dtype=DTYPE)
class_weights = torch.tensor([1.0, 0.5, 2.0], dtype=DTYPE)
_register_case("ex053", {'losses': losses, 'class_weights': class_weights})


In [ ]:
# Exercise 053: complete only the fields requested above; do not use autograd.
# Define `ex053_losses_shape`.
# Define `ex053_class_weights_shape`.
# Define `ex053_out_shape`.
# Define `ex053_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex053_losses_shape", "ex053", "losses_shape")
_check_private_value("ex053_class_weights_shape", "ex053", "class_weights_shape")
_check_private_value("ex053_out_shape", "ex053", "out_shape")
_check_private_tensor("ex053_out", "ex053", "out")


### Exercise 054 — Image channel bias

**Purpose:** Prepare one added value for each channel in images shaped `(batch, channel, height, width)`.

**Visible inputs:** `images`, `bias`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `bias_view + images`

**Task:** Create `bias_view = bias[None, :, None, None]`, then add it to each image location.

**Ingredients:** Axes are `(batch, channel, height, width)`.

**Axis meaning in this exercise:** `images` uses `(batch, channel, height, width)`; the bias vector values belong to channels.

**New operation or idea explained:** A **channel bias** is one added value for each image channel. The same channel value is reused for every batch item, height position, and width position.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Required predictions and outputs:**

- `ex054_images_shape`: a literal Python shape tuple
- `ex054_bias_shape`: a literal Python shape tuple
- `ex054_bias_view_shape`: a literal Python shape tuple
- `ex054_bias_view`: the requested PyTorch tensor
- `ex054_out_shape`: a literal Python shape tuple
- `ex054_out`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Attention mask across batches and heads


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
images = torch.arange(48, dtype=DTYPE).reshape(2, 3, 2, 4)
bias = torch.tensor([0.1, 0.2, 0.3], dtype=DTYPE)
_register_case("ex054", {'images': images, 'bias': bias})


In [ ]:
# Exercise 054: complete only the fields requested above; do not use autograd.
# Define `ex054_images_shape`.
# Define `ex054_bias_shape`.
# Define `ex054_bias_view_shape`.
# Define `ex054_bias_view` with PyTorch tensor operations.
# Define `ex054_out_shape`.
# Define `ex054_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex054_images_shape", "ex054", "images_shape")
_check_private_value("ex054_bias_shape", "ex054", "bias_shape")
_check_private_value("ex054_bias_view_shape", "ex054", "bias_view_shape")
_check_private_tensor("ex054_bias_view", "ex054", "bias_view")
_check_private_value("ex054_out_shape", "ex054", "out_shape")
_check_private_tensor("ex054_out", "ex054", "out")


### Exercise 055 — Attention mask across batches and heads

**Purpose:** Use one query-by-key mask for every batch item and every attention head.

**Visible inputs:** `scores`, `mask`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `scores.masked_fill(~mask, -1e9)`

**Task:** Mask future key positions in every batch and head.

**Ingredients:** Score axes are `(batch, heads, query, key)`; mask axes are `(query, key)`.

**Axis meaning in this exercise:** `scores` uses `(batch, heads, query, key)`; `mask` uses `(query, key)`.

**New operation or idea explained:** An **attention score** measures how strongly one query position relates to one key position. The Boolean mask says which query-key pairs are allowed. `~mask` flips `True` and `False`; `masked_fill(condition, value)` replaces positions where `condition` is `True`.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Explicit operator-ready tensors:** Before calculating the output, write the complete tensor values PyTorch behaves as if each operand has at the common elementwise shape. These are reasoning tensors; PyTorch usually does not allocate these full copies. Define each requested field with a literal `torch.tensor(...)`, not with `expand`, `broadcast_to`, `repeat`, or the original variable. Use the operand's dtype (`DTYPE` for floating tensors and `torch.bool` for Boolean tensors).

**Required predictions and outputs:**

- `ex055_scores_before_op`: an explicit `torch.tensor(...)` literal showing `scores` at the common operator-ready shape
- `ex055_inverted_mask_before_op`: an explicit `torch.tensor(...)` literal showing `~mask` at the common operator-ready shape

- `ex055_scores_shape`: a literal Python shape tuple
- `ex055_mask_shape`: a literal Python shape tuple
- `ex055_out_shape`: a literal Python shape tuple
- `ex055_out`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Pairwise feature differences


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
scores = torch.arange(18, dtype=DTYPE).reshape(1, 2, 3, 3)
mask = torch.tensor([[True, False, False], [True, True, False], [True, True, True]], dtype=torch.bool)
_register_case("ex055", {'scores': scores, 'mask': mask})


In [ ]:
# Exercise 055: complete only the fields requested above; do not use autograd.
# Define `ex055_scores_shape`.
# Define `ex055_mask_shape`.
# Define `ex055_out_shape`.
# Define `ex055_out` with PyTorch tensor operations.
# Define `ex055_scores_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex055_inverted_mask_before_op` as an explicit Boolean `torch.tensor(...)` literal for `~mask`.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex055_scores_shape", "ex055", "scores_shape")
_check_private_value("ex055_mask_shape", "ex055", "mask_shape")
_check_private_tensor("ex055_scores_before_op", "ex055", "scores_before_op")
_check_private_tensor("ex055_inverted_mask_before_op", "ex055", "inverted_mask_before_op")
_check_private_value("ex055_out_shape", "ex055", "out_shape")
_check_private_tensor("ex055_out", "ex055", "out")


### Exercise 056 — Pairwise feature differences

**Purpose:** Add size-one item axes so every vector in `x` can be compared with every vector in `y`.

**Visible inputs:** `x`, `y`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `x_rows - y_rows`

**Task:** Create `x_rows = x[:, None, :]` and `y_rows = y[None, :, :]`, then compute every feature-wise pair difference.

**Ingredients:** Prepared axes are `(x_items, 1, features)` and `(1, y_items, features)`.

**Axis meaning in this exercise:** `x` uses `(x_items, features)` and `y` uses `(y_items, features)`.

**New operation or idea explained:** **Pairwise** means every item from `x` is compared with every item from `y`. A feature-wise difference keeps the feature axis instead of summing it yet.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Explicit operator-ready tensors:** Before calculating the output, write the complete tensor values PyTorch behaves as if each operand has at the common elementwise shape. These are reasoning tensors; PyTorch usually does not allocate these full copies. Define each requested field with a literal `torch.tensor(...)`, not with `expand`, `broadcast_to`, `repeat`, or the original variable. Use the operand's dtype (`DTYPE` for floating tensors and `torch.bool` for Boolean tensors).

**Required predictions and outputs:**

- `ex056_x_rows_before_op`: an explicit `torch.tensor(...)` literal showing `x_rows` at the common operator-ready shape
- `ex056_y_rows_before_op`: an explicit `torch.tensor(...)` literal showing `y_rows` at the common operator-ready shape

- `ex056_x_shape`: a literal Python shape tuple
- `ex056_y_shape`: a literal Python shape tuple
- `ex056_x_rows_shape`: a literal Python shape tuple
- `ex056_x_rows`: the requested PyTorch tensor
- `ex056_y_rows_shape`: a literal Python shape tuple
- `ex056_y_rows`: the requested PyTorch tensor
- `ex056_out_shape`: a literal Python shape tuple
- `ex056_out`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Expand a row explicitly


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
x = torch.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0]], dtype=DTYPE)
y = torch.tensor([[1.0, 1.0, 1.0], [2.0, 2.0, 2.0], [3.0, 3.0, 3.0]], dtype=DTYPE)
_register_case("ex056", {'x': x, 'y': y})


In [ ]:
# Exercise 056: complete only the fields requested above; do not use autograd.
# Define `ex056_x_shape`.
# Define `ex056_y_shape`.
# Define `ex056_x_rows_shape`.
# Define `ex056_x_rows` with PyTorch tensor operations.
# Define `ex056_y_rows_shape`.
# Define `ex056_y_rows` with PyTorch tensor operations.
# Define `ex056_out_shape`.
# Define `ex056_out` with PyTorch tensor operations.
# Define `ex056_x_rows_before_op` as an explicit `torch.tensor(...)` literal.
# Define `ex056_y_rows_before_op` as an explicit `torch.tensor(...)` literal.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex056_x_shape", "ex056", "x_shape")
_check_private_value("ex056_y_shape", "ex056", "y_shape")
_check_private_value("ex056_x_rows_shape", "ex056", "x_rows_shape")
_check_private_tensor("ex056_x_rows", "ex056", "x_rows")
_check_private_value("ex056_y_rows_shape", "ex056", "y_rows_shape")
_check_private_tensor("ex056_y_rows", "ex056", "y_rows")
_check_private_tensor("ex056_x_rows_before_op", "ex056", "x_rows_before_op")
_check_private_tensor("ex056_y_rows_before_op", "ex056", "y_rows_before_op")
_check_private_value("ex056_out_shape", "ex056", "out_shape")
_check_private_tensor("ex056_out", "ex056", "out")


## 8. Explicit APIs, storage, and capstones

Use expand/repeat deliberately, then integrate broadcasting with reductions and realistic tensor programs.

This section covers Exercises 057–064.


### Exercise 057 — Expand a row explicitly

**Purpose:** Use `expand_as` to make a same-storage view that behaves like the target shape.

**Visible inputs:** `source`, `target`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `source.expand_as(target)`

**Task:** Expand the one-row source to the target shape.

**Ingredients:** `expand_as` may enlarge singleton dimensions; it does not accept incompatible non-singleton sizes.

**Axis meaning in this exercise:** For `source` and `target`, axis `0` means rows and axis `1` means columns.

**New operation or idea explained:** `source.expand_as(target)` returns a view of `source` that behaves as if it had `target.shape`. Only size-one axes may grow; repeated values are normally not copied.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.
- **View:** a tensor with different shape information that still refers to the same stored values.

**Required predictions and outputs:**

- `ex057_source_shape`: a literal Python shape tuple
- `ex057_target_shape`: a literal Python shape tuple
- `ex057_out_shape`: a literal Python shape tuple
- `ex057_out`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Broadcast to a target shape


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
source = torch.tensor([[1.0, 2.0, 3.0]], dtype=DTYPE)
target = torch.empty((2, 3), dtype=DTYPE)
_register_case("ex057", {'source': source, 'target': target})


In [ ]:
# Exercise 057: complete only the fields requested above; do not use autograd.
# Define `ex057_source_shape`.
# Define `ex057_target_shape`.
# Define `ex057_out_shape`.
# Define `ex057_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex057_source_shape", "ex057", "source_shape")
_check_private_value("ex057_target_shape", "ex057", "target_shape")
_check_private_value("ex057_out_shape", "ex057", "out_shape")
_check_private_tensor("ex057_out", "ex057", "out")


### Exercise 058 — Broadcast to a target shape

**Purpose:** Use `torch.broadcast_to` with the same rules used by elementwise operations.

**Visible inputs:** `source`, `target`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `torch.broadcast_to(source, target.shape)`

**Task:** Broadcast the source to `target.shape`.

**Ingredients:** The target supplies only a shape; its uninitialized values are irrelevant.

**Axis meaning in this exercise:** The source vector axis means columns; `target` uses `(rows, columns)`.

**New operation or idea explained:** `torch.broadcast_to(source, target.shape)` returns a broadcasted view with the requested shape. `target` supplies only its shape; values created by `torch.empty` are intentionally uninitialized and must not be read.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.
- **View:** a tensor with different shape information that still refers to the same stored values.

**Required predictions and outputs:**

- `ex058_source_shape`: a literal Python shape tuple
- `ex058_target_shape`: a literal Python shape tuple
- `ex058_out_shape`: a literal Python shape tuple
- `ex058_out`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Repeat materializes values


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
source = torch.tensor([1.0, 2.0, 3.0], dtype=DTYPE)
target = torch.empty((2, 3), dtype=DTYPE)
_register_case("ex058", {'source': source, 'target': target})


In [ ]:
# Exercise 058: complete only the fields requested above; do not use autograd.
# Define `ex058_source_shape`.
# Define `ex058_target_shape`.
# Define `ex058_out_shape`.
# Define `ex058_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex058_source_shape", "ex058", "source_shape")
_check_private_value("ex058_target_shape", "ex058", "target_shape")
_check_private_value("ex058_out_shape", "ex058", "out_shape")
_check_private_tensor("ex058_out", "ex058", "out")


### Exercise 059 — Repeat materializes values

**Purpose:** Compare copied repetition with a broadcasted view that reuses stored values.

**Visible inputs:** `source`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `source.repeat(2, 1)`

**Task:** Repeat the row twice.

**Ingredients:** `repeat(2, 1)` repeats axis `0` twice and axis `1` once.

**Axis meaning in this exercise:** For `source`, axis `0` means rows and axis `1` means columns.

**New operation or idea explained:** `repeat(2, 1)` physically repeats values: twice along axis `0` and once along axis `1`. Unlike broadcasting, `repeat` allocates repeated data.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Required predictions and outputs:**

- `ex059_source_shape`: a literal Python shape tuple
- `ex059_out_shape`: a literal Python shape tuple
- `ex059_out`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Expanded view shares storage


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
source = torch.tensor([[1.0, 2.0, 3.0]], dtype=DTYPE)
_register_case("ex059", {'source': source})


In [ ]:
# Exercise 059: complete only the fields requested above; do not use autograd.
# Define `ex059_source_shape`.
# Define `ex059_out_shape`.
# Define `ex059_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex059_source_shape", "ex059", "source_shape")
_check_private_value("ex059_out_shape", "ex059", "out_shape")
_check_private_tensor("ex059_out", "ex059", "out")


### Exercise 060 — Expanded view shares storage

**Purpose:** See that an expanded view and its source can refer to the same stored values.

**Visible inputs:** `source`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `source.expand(2, 3)`

**Task:** Create the expanded view and predict whether it shares storage with `source`.

**Ingredients:** `expand` normally changes strides rather than copying values. This storage topic is optional for basic broadcasting use.

**Axis meaning in this exercise:** For `source`, axis `0` means rows and axis `1` means columns.

**New operation or idea explained:** `expand(2, 3)` returns a view that behaves like shape `(2, 3)` while reusing the source’s stored values. A **stride** controls movement through storage; stride zero rereads the same value.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.
- **View:** a tensor with different shape information that still refers to the same stored values.
- **Storage:** the memory holding tensor values. **Sharing storage** means two tensors refer to that same memory.

**Required predictions and outputs:**

- `ex060_source_shape`: a literal Python shape tuple
- `ex060_out_shape`: a literal Python shape tuple
- `ex060_shares_storage`: a Python `bool`
- `ex060_out`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Clone materializes an expanded view


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
source = torch.tensor([[1.0, 2.0, 3.0]], dtype=DTYPE)
_register_case("ex060", {'source': source})


In [ ]:
# Exercise 060: complete only the fields requested above; do not use autograd.
# Define `ex060_source_shape`.
# Define `ex060_out_shape`.
# Define `ex060_shares_storage`.
# Define `ex060_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex060_source_shape", "ex060", "source_shape")
_check_private_value("ex060_out_shape", "ex060", "out_shape")
_check_private_value("ex060_shares_storage", "ex060", "shares_storage")
_check_private_tensor("ex060_out", "ex060", "out")


### Exercise 061 — Clone materializes an expanded view

**Purpose:** Turn a broadcasted view into independent storage.

**Visible inputs:** `source`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `source.expand(2, 3).clone()`

**Task:** Expand, clone, and predict storage sharing.

**Ingredients:** Cloning allocates independent storage for the visible values.

**Axis meaning in this exercise:** For `source`, axis `0` means rows and axis `1` means columns.

**New operation or idea explained:** `clone()` creates a new tensor with its own stored values. After `expand(...).clone()`, the visible repeated values no longer share storage with the source.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.
- **View:** a tensor with different shape information that still refers to the same stored values.
- **Storage:** the memory holding tensor values. **Sharing storage** means two tensors refer to that same memory.

**Required predictions and outputs:**

- `ex061_source_shape`: a literal Python shape tuple
- `ex061_out_shape`: a literal Python shape tuple
- `ex061_shares_storage`: a Python `bool`
- `ex061_out`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Final practice: column normalization


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
source = torch.tensor([[1.0, 2.0, 3.0]], dtype=DTYPE)
_register_case("ex061", {'source': source})


In [ ]:
# Exercise 061: complete only the fields requested above; do not use autograd.
# Define `ex061_source_shape`.
# Define `ex061_out_shape`.
# Define `ex061_shares_storage`.
# Define `ex061_out` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex061_source_shape", "ex061", "source_shape")
_check_private_value("ex061_out_shape", "ex061", "out_shape")
_check_private_value("ex061_shares_storage", "ex061", "shares_storage")
_check_private_tensor("ex061_out", "ex061", "out")


### Exercise 062 — Final practice: column normalization

**Purpose:** Combine column means, subtraction, scaling, and broadcasting in one final multi-step exercise.

**Visible inputs:** `x`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `normalized`

**Task:** Compute every named intermediate and normalize each feature column.

**Ingredients:** Use `mean = x.mean(dim=0, keepdim=True)`, `centered = x - mean`, `variance = (centered ** 2).mean(dim=0, keepdim=True)`, `inv_std = (variance + 1e-5).rsqrt()`, and `normalized = centered * inv_std`. This uses population variance and epsilon `1e-5`.

**Axis meaning in this exercise:** For `x`, axis `0` means examples and axis `1` means features; each column is one feature.

**New operation or idea explained:** **Normalization** recenters and rescales values. `mean(dim=0, keepdim=True)` averages examples while keeping a size-one example axis. **Variance** is the mean squared distance from the mean. **Epsilon** (`1e-5`) prevents division by zero. `rsqrt()` computes one divided by the square root.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.
- **Reduction:** an operation such as `mean` or `sum` that combines several values into fewer values.

**Required predictions and outputs:**

- `ex062_x_shape`: a literal Python shape tuple
- `ex062_mean_shape`: a literal Python shape tuple
- `ex062_mean`: the requested PyTorch tensor
- `ex062_centered_shape`: a literal Python shape tuple
- `ex062_centered`: the requested PyTorch tensor
- `ex062_variance_shape`: a literal Python shape tuple
- `ex062_variance`: the requested PyTorch tensor
- `ex062_inv_std_shape`: a literal Python shape tuple
- `ex062_inv_std`: the requested PyTorch tensor
- `ex062_normalized_shape`: a literal Python shape tuple
- `ex062_normalized`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Final practice: attention masking


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
x = torch.tensor([[1.0, 2.0, 3.0], [2.0, 4.0, 6.0], [3.0, 6.0, 9.0], [4.0, 8.0, 12.0]], dtype=DTYPE)
_register_case("ex062", {'x': x})


In [ ]:
# Exercise 062: complete only the fields requested above; do not use autograd.
# Define `ex062_x_shape`.
# Define `ex062_mean_shape`.
# Define `ex062_mean` with PyTorch tensor operations.
# Define `ex062_centered_shape`.
# Define `ex062_centered` with PyTorch tensor operations.
# Define `ex062_variance_shape`.
# Define `ex062_variance` with PyTorch tensor operations.
# Define `ex062_inv_std_shape`.
# Define `ex062_inv_std` with PyTorch tensor operations.
# Define `ex062_normalized_shape`.
# Define `ex062_normalized` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex062_x_shape", "ex062", "x_shape")
_check_private_value("ex062_mean_shape", "ex062", "mean_shape")
_check_private_tensor("ex062_mean", "ex062", "mean")
_check_private_value("ex062_centered_shape", "ex062", "centered_shape")
_check_private_tensor("ex062_centered", "ex062", "centered")
_check_private_value("ex062_variance_shape", "ex062", "variance_shape")
_check_private_tensor("ex062_variance", "ex062", "variance")
_check_private_value("ex062_inv_std_shape", "ex062", "inv_std_shape")
_check_private_tensor("ex062_inv_std", "ex062", "inv_std")
_check_private_value("ex062_normalized_shape", "ex062", "normalized_shape")
_check_private_tensor("ex062_normalized", "ex062", "normalized")


### Exercise 063 — Final practice: attention masking

**Purpose:** Prepare a causal mask and apply it across batch and head axes.

**Visible inputs:** `scores`, `causal`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `masked_scores`

**Task:** Create the mask view and masked scores.

**Ingredients:** Use `mask_view = causal[None, None, :, :]` and `masked_scores = scores.masked_fill(~mask_view, -1e9)`. Axes are `(batch, heads, query, key)`.

**Axis meaning in this exercise:** `scores` uses `(batch, heads, query, key)`; `causal` uses `(query, key)`.

**New operation or idea explained:** A **causal mask** prevents a query position from using future key positions. `masked_fill(~mask_view, -1e9)` puts a very negative value wherever attention is forbidden.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.

**Required predictions and outputs:**

- `ex063_scores_shape`: a literal Python shape tuple
- `ex063_causal_shape`: a literal Python shape tuple
- `ex063_mask_view_shape`: a literal Python shape tuple
- `ex063_mask_view`: the requested PyTorch tensor
- `ex063_masked_scores_shape`: a literal Python shape tuple
- `ex063_masked_scores`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Final practice: pairwise squared distances


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
scores = torch.arange(64, dtype=DTYPE).reshape(2, 2, 4, 4)
causal = torch.tensor([[True, False, False, False], [True, True, False, False], [True, True, True, False], [True, True, True, True]], dtype=torch.bool)
_register_case("ex063", {'scores': scores, 'causal': causal})


In [ ]:
# Exercise 063: complete only the fields requested above; do not use autograd.
# Define `ex063_scores_shape`.
# Define `ex063_causal_shape`.
# Define `ex063_mask_view_shape`.
# Define `ex063_mask_view` with PyTorch tensor operations.
# Define `ex063_masked_scores_shape`.
# Define `ex063_masked_scores` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex063_scores_shape", "ex063", "scores_shape")
_check_private_value("ex063_causal_shape", "ex063", "causal_shape")
_check_private_value("ex063_mask_view_shape", "ex063", "mask_view_shape")
_check_private_tensor("ex063_mask_view", "ex063", "mask_view")
_check_private_value("ex063_masked_scores_shape", "ex063", "masked_scores_shape")
_check_private_tensor("ex063_masked_scores", "ex063", "masked_scores")


### Exercise 064 — Final practice: pairwise squared distances

**Purpose:** Compare every pair of items, keep feature values separate, then sum across features.

**Visible inputs:** `x`, `y`. Run the fixture below and read its construction code; it deliberately prints no expected answers.

**Operation to reason about:** `sq_distance`

**Task:** Create item-axis views, compute differences, square, and sum the feature axis.

**Ingredients:** Use `x_rows = x[:, None, :]`, `y_rows = y[None, :, :]`, `difference = x_rows - y_rows`, and `sq_distance = (difference ** 2).sum(dim=2)`.

**Axis meaning in this exercise:** `x` uses `(x_items, features)` and `y` uses `(y_items, features)`; the final sum combines feature values.

**New operation or idea explained:** A **squared distance** between two vectors is found by subtracting their features, squaring each difference, and summing those squares across the feature axis.

**Beginner definitions used here:**

- **Operand:** an input tensor used by the operator. In `a + b`, `a` and `b` are operands.
- **Axis:** one direction in a tensor's shape, numbered from left to right starting at `0`.
- **Singleton axis:** an axis with size `1`; its one value may be reused along that direction.
- **Reduction:** an operation such as `mean` or `sum` that combines several values into fewer values.

**Required predictions and outputs:**

- `ex064_x_shape`: a literal Python shape tuple
- `ex064_y_shape`: a literal Python shape tuple
- `ex064_x_rows_shape`: a literal Python shape tuple
- `ex064_x_rows`: the requested PyTorch tensor
- `ex064_y_rows_shape`: a literal Python shape tuple
- `ex064_y_rows`: the requested PyTorch tensor
- `ex064_difference_shape`: a literal Python shape tuple
- `ex064_difference`: the requested PyTorch tensor
- `ex064_sq_distance_shape`: a literal Python shape tuple
- `ex064_sq_distance`: the requested PyTorch tensor

**Order:** Fill the requested shape reasoning before calculating tensor values. Only derive aligned shapes or expanded axes when this exercise explicitly asks for them.

**Next concept:** Fresh-kernel mastery run


In [ ]:
# Supplied visible fixture: inspect these values; it prints no expected answers.
x = torch.tensor([[0.0, 0.0, 0.0], [1.0, 2.0, 3.0]], dtype=DTYPE)
y = torch.tensor([[1.0, 0.0, 0.0], [0.0, 2.0, 0.0], [0.0, 0.0, 3.0]], dtype=DTYPE)
_register_case("ex064", {'x': x, 'y': y})


In [ ]:
# Exercise 064: complete only the fields requested above; do not use autograd.
# Define `ex064_x_shape`.
# Define `ex064_y_shape`.
# Define `ex064_x_rows_shape`.
# Define `ex064_x_rows` with PyTorch tensor operations.
# Define `ex064_y_rows_shape`.
# Define `ex064_y_rows` with PyTorch tensor operations.
# Define `ex064_difference_shape`.
# Define `ex064_difference` with PyTorch tensor operations.
# Define `ex064_sq_distance_shape`.
# Define `ex064_sq_distance` with PyTorch tensor operations.
# Write your work below, then run the supplied test cell.


In [ ]:
# Supplied test: expected shapes and values remain private.
_check_private_value("ex064_x_shape", "ex064", "x_shape")
_check_private_value("ex064_y_shape", "ex064", "y_shape")
_check_private_value("ex064_x_rows_shape", "ex064", "x_rows_shape")
_check_private_tensor("ex064_x_rows", "ex064", "x_rows")
_check_private_value("ex064_y_rows_shape", "ex064", "y_rows_shape")
_check_private_tensor("ex064_y_rows", "ex064", "y_rows")
_check_private_value("ex064_difference_shape", "ex064", "difference_shape")
_check_private_tensor("ex064_difference", "ex064", "difference")
_check_private_value("ex064_sq_distance_shape", "ex064", "sq_distance_shape")
_check_private_tensor("ex064_sq_distance", "ex064", "sq_distance")


## Completion standard

You have mastered this notebook when you can:

- distinguish scalar, vector, matrix, and higher-rank shapes;
- explain broadcasting first as value reuse;
- use singleton dimensions to express row, column, feature, batch, token, and channel intent;
- right-align shapes when debugging compatibility;
- distinguish an aligned shape from a tuple of expanded axis numbers;
- predict result shapes without trial-and-error reshaping;
- diagnose invalid operations and repair them deliberately;
- use `expand`, `broadcast_to`, `repeat`, and `clone` for the intended storage behavior;
- apply broadcasting in common machine-learning tensor layouts.

Manual gradient derivation is intentionally omitted from this beginner workbook. Continue with the paired tensor-shapes/manual-backpropagation notebook after broadcasting itself is comfortable.

## Official references

- [PyTorch broadcasting semantics](https://docs.pytorch.org/docs/stable/notes/broadcasting.html)
- [`Tensor.unsqueeze`](https://docs.pytorch.org/docs/stable/generated/torch.unsqueeze.html)
- [`Tensor.expand`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.expand.html)
- [`torch.broadcast_to`](https://docs.pytorch.org/docs/stable/generated/torch.broadcast_to.html)
- [`Tensor.repeat`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.repeat.html)
- [Tensor views](https://docs.pytorch.org/docs/stable/tensor_view.html)
